In [2]:
# =======================================================================
# Cell 1 - CONFIG, IMPORTS & FOLDER STRUCTURE
# =======================================================================

import os, sys, gc, copy, glob, math, time, json, pickle, hashlib, warnings
from contextlib import contextmanager
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from scipy.special import expit

# ML
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit, learning_curve
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, log_loss,
    cohen_kappa_score, matthews_corrcoef, mean_absolute_error,
    mean_squared_error, precision_recall_curve, precision_score,
    r2_score, recall_score, roc_auc_score, roc_curve, brier_score_loss,
)

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.labelsize': 12, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

PAPER_COLORS = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
]

try:
    import shap
except Exception:
    shap = None

try:
    import joblib
except Exception:
    joblib = None

# -----------------------------------------------------------------------
# Utility Functions
# -----------------------------------------------------------------------
def join_if_root(root, child):
    return str(Path(root) / child) if root else None

def resolve_existing_dir(candidates):
    for c in candidates:
        if c and Path(c).exists():
            return str(c)
    return None

@contextmanager
def cell_timer(name):
    t0 = time.time()
    print(f'\n  {name} started...')
    yield
    print(f'  {name} completed in {time.time() - t0:.1f}s')

STOCKS_PATH = resolve_existing_dir([
    os.getenv('STOCKS_PATH'),
    join_if_root(os.getenv('REAL_WORLD_STOCK_DATA_ROOT'), 'Stocks'),
    join_if_root(os.getenv('STOCK_DATA_ROOT'), 'Stocks'),
    '/kaggle/input/datasets/borismarjanovic/price-volume-data-for-all-us-stocks-etfs/Stocks',
])

ETF_PATH = resolve_existing_dir([
    os.getenv('ETF_PATH'),
    join_if_root(os.getenv('REAL_WORLD_STOCK_DATA_ROOT'), 'ETFs'),
    join_if_root(os.getenv('STOCK_DATA_ROOT'), 'ETFs'),
    '/kaggle/input/datasets/borismarjanovic/price-volume-data-for-all-us-stocks-etfs/ETFs',
])

CONFIG = {
    # Data paths
    'stocks_path': STOCKS_PATH,
    'etf_path': ETF_PATH,
    'max_stock_files': 400,
    'max_etf_files': 50,
    'top_n_tickers': 24,
    'min_history_days': 252,
    'min_avg_dollar_volume': 2_500_000,
    # Sequence
    'seq_len': 20,
    'batch_size': 256,
    # DL training  [NEW: gradient clipping, LR scheduling, early stopping]
    'lstm_epochs': 25,
    'tcn_epochs': 25,
    'transformer_epochs': 25,
    'dl_patience': 6,       # early-stopping patience
    'dl_lr': 3e-4,          # base learning rate
    'dl_weight_decay': 1e-4,# L2 regularisation
    'dl_grad_clip': 1.0,    # gradient clipping max-norm
    # Labels
    'future_horizon': 1,
    'target_return_threshold': 0.0015,
    'adaptive_threshold': True,
    'label_smoothing': 0.05,
    # Walk-forward  [NEW]
    'wf_n_folds': 5,
    'wf_min_train_days': 252,
    # Backtesting
    'transaction_cost': 0.0005,
    'slippage_factor': 0.0002,
    'starting_capital': 100_000,
    'max_position_concentration': 0.10,
    'drawdown_reduction_threshold': -0.10,
    'drawdown_reduction_factor': 0.50,
    'kelly_fraction': 0.25, # [NEW] fractional Kelly multiplier
    'max_daily_turnover': 0.40, # [NEW] turnover cap
    # Train/Val/Test split
    'split_method': 'chronological',
    'train_fraction': 0.70,
    'val_fraction': 0.15,
    'tscv_folds': 4,
    'embargo_days': 3,
    # Feature selection
    'max_features_modeling': 55,
    'feature_corr_threshold': 0.97,
    'winsorize_percentile': 0.01,
    # Strategies
    'long_only_top_pct': 0.30,
    'long_short_top_pct': 0.15,
    'long_short_bottom_pct': 0.15,
    'long_short_smoothing_span': 5,
    'long_short_target_vol': 0.02,
    'buy_signal_threshold': 0.55,
    'sell_signal_threshold': 0.45,
    # Regime detection  [NEW]
    'n_regimes': 3,
    'regime_lookback': 60,
    # Model selection  [NEW: IC weight added]
    'enable_dl_models': True,
    'threshold_metric': 'mcc',
    'selection_weights': {
        'val_auc': 0.25,
        'val_pr_auc': 0.15,
        'val_sharpe': 0.30,
        'val_ic': 0.15,     # [NEW] Information Coefficient
        'val_cumulative_return': 0.15,
    },
    # Explainability
    'shap_sample_size': 1500,
    'shap_background_size': 300,
    # Signal decay  [NEW]
    'ic_horizons': [1, 2, 5, 10, 20],
    # Misc
    'random_seed': 42,
    'scaler_type': 'robust',
    'output_root': 'artifacts',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

def validate_config(cfg):
    assert cfg['train_fraction'] + cfg['val_fraction'] < 1.0
    assert cfg['future_horizon'] >= 1 and cfg['seq_len'] >= 5
    assert 0 < cfg['target_return_threshold'] < 0.1
    assert cfg['buy_signal_threshold'] > cfg['sell_signal_threshold']
    assert 0 < cfg['max_position_concentration'] <= 1.0
    assert 0 < cfg['kelly_fraction'] <= 1.0
    print('Config validation passed')

validate_config(CONFIG)

if not CONFIG['stocks_path'] or not CONFIG['etf_path']:
    raise FileNotFoundError(
        'Cannot find stock/ETF data. Set STOCKS_PATH & ETF_PATH env vars, '
        'or place Stooq-style data under D:/ML Lab/projects/dataset/Data/.'
    )

np.random.seed(CONFIG['random_seed'])
torch.manual_seed(CONFIG['random_seed'])
if CONFIG['device'] == 'cuda':
    torch.cuda.manual_seed_all(CONFIG['random_seed'])

OUTPUT_DIRS = {}
for _name in ['root', 'models', 'plots', 'reports', 'predictions', 'monitoring', 'backtests', 'walkforward']:
    _p = Path(CONFIG['output_root']) / _name if _name != 'root' else Path(CONFIG['output_root'])
    _p.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIRS[_name] = _p

def save_plot(name, fig=None):
    """Save current figure then close it to prevent memory leaks."""
    try:
        if fig is None:
            fig = plt.gcf()
        fig.tight_layout()
        fig.savefig(OUTPUT_DIRS['plots'] / name, dpi=300, bbox_inches='tight', facecolor='white')
    except Exception as e:
        print(f"Error saving {name}: {e}")
    finally:
        if fig is not None:
            plt.close(fig)
        else:
            plt.close()

def save_pickle_artifact(path, obj):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def save_json_artifact(path, obj):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=str)

def get_scaler(name):
    n = name.lower()
    if n == 'robust': return RobustScaler(quantile_range=(10, 90))
    if n == 'standard': return StandardScaler()
    return MinMaxScaler()

def compute_data_hash(df, config):
    h = hashlib.sha256()
    h.update(str(df.shape).encode())
    h.update(str(sorted(df.columns.tolist())).encode())
    h.update(str(sorted(config.items())).encode())
    if len(df):
        h.update(df.iloc[0].to_string().encode())
        h.update(df.iloc[-1].to_string().encode())
    return h.hexdigest()[:16]

# -----------------------------------------------------------------------
# Information Coefficient (IC) / Information Ratio (IR)
# These are THE real-world quant metrics for signal quality
# -----------------------------------------------------------------------
def information_coefficient(scores, returns):
    """Spearman rank IC between predicted scores and realized returns."""
    s = np.asarray(scores, dtype=float)
    r = np.asarray(returns, dtype=float)
    valid = ~(np.isnan(s) | np.isnan(r) | np.isinf(s) | np.isinf(r))
    if valid.sum() < 10:
        return 0.0
    ic, _ = spearmanr(s[valid], r[valid])
    return float(ic) if not np.isnan(ic) else 0.0

def information_ratio(ic_series):
    """IR = mean(IC) / std(IC) -- persistence-adjusted signal quality."""
    ics = np.asarray(ic_series, dtype=float)
    ics = ics[~np.isnan(ics)]
    if len(ics) < 3 or np.std(ics) < 1e-9:
        return 0.0
    return float(ics.mean() / ics.std())

# -----------------------------------------------------------------------
# Risk / Tail-Risk Utilities
# -----------------------------------------------------------------------
def cvar(returns, alpha=0.05):
    """Conditional Value at Risk (Expected Shortfall)."""
    arr = np.asarray(returns, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return 0.0
    cutoff = np.percentile(arr, alpha * 100)
    tail = arr[arr <= cutoff]
    return float(tail.mean()) if len(tail) else float(cutoff)

def omega_ratio(returns, threshold=0.0):
    """Omega ratio: probability-weighted upside / downside."""
    arr = np.asarray(returns, dtype=float)
    arr = arr[~np.isnan(arr)]
    gains = arr[arr > threshold] - threshold
    losses = threshold - arr[arr <= threshold]
    return float(gains.sum() / losses.sum()) if losses.sum() > 0 else float('inf')

def compute_kelly(win_rate, avg_win, avg_loss):
    """Full Kelly fraction (apply CONFIG['kelly_fraction'] to scale)."""
    if avg_loss < 1e-9 or win_rate <= 0 or win_rate >= 1:
        return 0.0
    b = avg_win / (avg_loss + 1e-9)
    return max(0.0, (b * win_rate - (1 - win_rate)) / (b + 1e-9))

_pipeline_start = time.time()
print(f'Device:          {CONFIG["device"]}')
print(f'Random seed:     {CONFIG["random_seed"]}')
print(f'Stocks path:     {CONFIG["stocks_path"]}')
print(f'ETF path:        {CONFIG["etf_path"]}')
print(f'DL patience:     {CONFIG["dl_patience"]}')
print(f'Walk-fwd folds:  {CONFIG["wf_n_folds"]}')
print(f'Kelly fraction:  {CONFIG["kelly_fraction"]}')
print(f'IC horizons:     {CONFIG["ic_horizons"]}')
print(f'SHAP available:  {shap is not None}')
print(f'joblib:          {joblib is not None}')


# =======================================================================
# Cell 2 - DATA LOADING & CLEANING
# =======================================================================

with cell_timer('Data Loading & Cleaning'):

    REQUIRED_PRICE_COLUMNS = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    PRICE_COLUMN_ALIASES = {
        'date':'Date', 'open':'Open', 'high':'High', 'low':'Low',
        'close':'Close', 'adjclose':'Close', 'adjustedclose':'Close', 'volume':'Volume',
    }

    def normalize_price_columns(df):
        rename_map = {}
        for col in df.columns:
            key = col.strip().lower().replace(' ', '').replace('_', '')
            if key in PRICE_COLUMN_ALIASES:
                rename_map[col] = PRICE_COLUMN_ALIASES[key]
        return df.rename(columns=rename_map)

    def infer_ticker_from_file(fp):
        return Path(fp).stem.upper().replace('.US', '')

    def read_price_file(fp):
        suffix = Path(fp).suffix.lower()
        try:
            if suffix == '.parquet':
                df = pd.read_parquet(fp)
            else:
                allowed = {'date', 'open', 'high', 'low', 'close', 'adj close', 'adj_close', 'adjclose', 'volume'}
                df = pd.read_csv(fp, usecols=lambda c: c.strip().lower() in allowed, low_memory=False)
        except Exception:
            return None

        if df is None or df.empty:
            return None

        df = normalize_price_columns(df)
        if not set(REQUIRED_PRICE_COLUMNS).issubset(df.columns):
            return None

        df = df[REQUIRED_PRICE_COLUMNS].copy()
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        for col in ['Open', 'High', 'Low', 'Close']:
            df[col] = pd.to_numeric(df[col], errors='coerce', downcast='float')
        df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce', downcast='integer')
        df = df.dropna(subset=REQUIRED_PRICE_COLUMNS)

        if df.empty or len(df) < CONFIG['min_history_days']:
            return None

        # ---- Data Quality Gate ----------------------------------------
        pct_chg = df['Close'].pct_change().abs()
        df = df[pct_chg < 2.0].reset_index(drop=True)          # drop >200% daily moves
        df = df[df['High'] >= df['Low']].reset_index(drop=True) # sanity: H >= L
        df = df[df['High'] >= df['Close']].reset_index(drop=True)
        df = df[df['Low']  <= df['Close']].reset_index(drop=True)

        if len(df) < CONFIG['min_history_days']:
            return None

        df['ticker'] = infer_ticker_from_file(fp)
        return df.sort_values('Date').reset_index(drop=True)

    def discover_price_files(path):
        if not path or not Path(path).exists():
            return []
        files = []
        for pattern in ('*.txt', '*.csv', '*.parquet'):
            files.extend(sorted(Path(path).glob(pattern)))
        return [str(p) for p in files]

    def load_files(path, limit, force_file=None):
        files = discover_price_files(path)[:limit]
        if force_file:
            for fp in [str(Path(path)/force_file),
                       str(Path(path)/force_file.replace('.txt', '.csv')),
                       str(Path(path)/force_file.replace('.csv', '.txt'))]:
                if Path(fp).exists() and fp not in files:
                    files.append(fp)

        dfs = []
        for f in files:
            df = read_price_file(f)
            if df is not None and not df.empty:
                dfs.append(df)

        if not dfs:
            return pd.DataFrame(columns=REQUIRED_PRICE_COLUMNS + ['ticker'])
        return pd.concat(dfs, ignore_index=True)

    print('Loading stock files...')
    stocks = load_files(CONFIG['stocks_path'], CONFIG['max_stock_files'])
    print('Loading ETF files...')
    etfs   = load_files(CONFIG['etf_path'], CONFIG['max_etf_files'], force_file='spy.us.txt')

    if stocks.empty and etfs.empty:
        raise ValueError('No input files loaded. Check CONFIG data paths.')

    raw_data = pd.concat([stocks, etfs], ignore_index=True)
    raw_data['Date'] = pd.to_datetime(raw_data['Date'], errors='coerce')
    raw_data = (raw_data.dropna(subset=['Date'])
                .sort_values(['ticker', 'Date'])
                .reset_index(drop=True))

    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        raw_data[col] = pd.to_numeric(raw_data[col], errors='coerce')

    raw_data.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'], inplace=True)

    # Remove penny stocks / zero-volume days
    raw_data = raw_data[(raw_data['Close'] > 0.5) & (raw_data['Volume'] > 0)].reset_index(drop=True)
    raw_data['dollar_volume'] = raw_data['Close'] * raw_data['Volume']

    # Liquidity filter
    avg_dv   = raw_data.groupby('ticker', observed=True)['dollar_volume'].mean()
    liquid   = avg_dv[avg_dv >= CONFIG['min_avg_dollar_volume']].index
    if len(liquid) > 0:
        raw_data = raw_data[raw_data['ticker'].isin(liquid)].reset_index(drop=True)

    # Ensure SPY is present
    if 'SPY' not in raw_data['ticker'].values:
        for alt in ['spy.us.txt', 'spy.txt', 'spy.csv']:
            alt_path = Path(CONFIG['etf_path']) / alt
            if alt_path.exists():
                spy_df = read_price_file(str(alt_path))
                if spy_df is not None and not spy_df.empty:
                    spy_df['ticker'] = 'SPY'
                    raw_data = pd.concat([raw_data, spy_df], ignore_index=True)
                    raw_data = raw_data.sort_values(['ticker', 'Date']).reset_index(drop=True)
                break

    DATA_HASH = compute_data_hash(raw_data, CONFIG)

    print(f'Raw data shape:   {raw_data.shape}')
    print(f'Unique tickers:   {raw_data["ticker"].nunique()}')
    print(f'Date range:       {raw_data["Date"].min().date()} to {raw_data["Date"].max().date()}')
    print(f'Data fingerprint: {DATA_HASH}')


# =======================================================================
# Cell 3 - FEATURE ENGINEERING (100+ Technical Indicators)
# =======================================================================

with cell_timer('Feature Engineering'):

    def trailing_zscore(series, window):
        mu    = series.rolling(window).mean()
        sigma = series.rolling(window).std()
        return (series - mu) / (sigma + 1e-9)

    def create_features(df):
        df = df.sort_values('Date').copy()
        c, h, l, o, v = (df['Close'], df['High'], df['Low'], df['Open'], df['Volume'])
        prev_c = c.shift(1)

        # == Price & Returns ==
        df['price_diff']      = c.diff()
        df['return']          = c.pct_change()
        df['log_return']      = np.log(c / prev_c)
        df['return_5d']       = c.pct_change(5)
        df['return_10d']      = c.pct_change(10)
        df['return_20d']      = c.pct_change(20)

        # == Moving Averages ==
        for w in [5, 10, 20, 50]:
            df[f'ma{w}']      = c.rolling(w).mean()
            df[f'ma_gap_{w}'] = c / (df[f'ma{w}'] + 1e-9) - 1
        df['ema_12']          = c.ewm(span=12, adjust=False).mean()
        df['ema_26']          = c.ewm(span=26, adjust=False).mean()
        df['ema_50']          = c.ewm(span=50, adjust=False).mean()
        df['trend_strength']  = df['ma10'] / (df['ma20'] + 1e-9) - 1
        df['ma_cross']        = (df['ema_12'] > df['ema_26']).astype(float)

        # == Volatility ==
        df['volatility_5']   = df['return'].rolling(5).std()
        df['volatility_10']  = df['return'].rolling(10).std()
        df['volatility_20']  = df['return'].rolling(20).std()
        df['volatility_60']  = df['return'].rolling(60).std()
        df['vol_regime']     = (df['volatility_20'] / (df['volatility_60'].rolling(30).mean() + 1e-9))
        
        # Parkinson realized volatility
        df['park_vol']       = np.sqrt((np.log(h / (l + 1e-9)) ** 2) / (4 * np.log(2) + 1e-9))

        # == Momentum ==
        for w in [5, 10, 20, 60]:
            df[f'momentum_{w}'] = c / (c.shift(w) + 1e-9) - 1

        # == Candlestick Features ==
        df['hl_range']          = (h - l) / (c + 1e-9)
        df['oc_gap']            = (o - prev_c) / (prev_c + 1e-9)
        df['intraday_return']   = (c - o) / (o + 1e-9)
        df['close_position']    = (c - l) / (h - l + 1e-9)
        df['candle_body']       = (c - o) / (h - l + 1e-9)
        df['upper_shadow']      = (h - np.maximum(c, o)) / (h - l + 1e-9)
        df['lower_shadow']      = (np.minimum(c, o) - l) / (h - l + 1e-9)
        df['doji']              = (df['hl_range'] > 0) & (df['candle_body'].abs() < 0.05)
        df['doji']              = df['doji'].astype(float)

        # == Lag Features ==
        for lag in [1, 2, 3, 5, 10]:
            df[f'return_lag_{lag}']   = df['return'].shift(lag)
        for lag in [1, 3, 5, 10]:
            df[f'volume_ratio_{lag}'] = v / (v.shift(lag) + 1e-9)

        # == RSI 14 & 28 ==
        for rsi_period in [14, 28]:
            delta = c.diff()
            gain  = delta.clip(lower=0).rolling(rsi_period).mean()
            loss  = (-delta.clip(upper=0)).rolling(rsi_period).mean()
            df[f'rsi_{rsi_period}'] = 100 - 100 / (1 + gain / (loss + 1e-9))

        # == MACD ==
        df['macd']        = df['ema_12'] - df['ema_26']
        df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
        df['macd_hist']   = df['macd'] - df['macd_signal']
        df['macd_norm']   = df['macd'] / (c + 1e-9)

        # == Stochastic ==
        low14  = l.rolling(14).min()
        high14 = h.rolling(14).max()
        df['stoch_k']   = 100 * (c - low14) / (high14 - low14 + 1e-9)
        df['stoch_d']   = df['stoch_k'].rolling(3).mean()
        df['stoch_rsi'] = df['rsi_14'].rolling(14).apply(
            lambda x: (x[-1] - x.min()) / (x.max() - x.min() + 1e-9), raw=True
        )

        # == Bollinger Bands ==
        bb_mid = c.rolling(20).mean()
        bb_std = c.rolling(20).std()
        df['bb_upper']      = bb_mid + 2 * bb_std
        df['bb_lower']      = bb_mid - 2 * bb_std
        df['bb_width']      = (df['bb_upper'] - df['bb_lower']) / (bb_mid + 1e-9)
        df['bb_percent_b']  = (c - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-9)

        # == ATR ==
        tr = pd.concat([(h-l), (h-prev_c).abs(), (l-prev_c).abs()], axis=1).max(axis=1)
        df['atr_14'] = tr.rolling(14).mean() / (c + 1e-9)
        df['atr_20'] = tr.rolling(20).mean() / (c + 1e-9)

        # == OBV ==
        obv_sign     = np.sign(c.diff()).fillna(0)
        obv          = (obv_sign * v).cumsum()
        df['obv_ratio'] = obv / (obv.rolling(10).mean() + 1e-9)

        # == Williams %R ==
        df['williams_r'] = -100 * (high14 - c) / (high14 - low14 + 1e-9)

        # == CCI ==
        tp     = (h + l + c) / 3
        tp_ma  = tp.rolling(20).mean()
        tp_std = tp.rolling(20).std()
        df['cci_20'] = (tp - tp_ma) / (0.015 * (tp_std * 0.8) + 1e-9)

        # == ADX ==
        plus_dm  = (h - h.shift(1)).clip(lower=0)
        minus_dm = (l.shift(1) - l).clip(lower=0)
        atr14    = tr.rolling(14).mean()
        plus_di  = 100 * (plus_dm.rolling(14).mean() / (atr14 + 1e-9))
        minus_di = 100 * (minus_dm.rolling(14).mean() / (atr14 + 1e-9))
        dx       = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
        df['adx_14']   = dx.rolling(14).mean()
        df['plus_di']  = plus_di
        df['minus_di'] = minus_di

        # == VWAP proxy ==
        df['vwap_ratio'] = c / (((h+l+c)/3 * v).rolling(20).sum() / (v.rolling(20).sum() + 1e-9) + 1e-9)

        # == Volume ==
        df['rel_volume']    = v / (v.rolling(20).mean() + 1e-9)
        df['dollar_volume'] = c * v
        df['dv_ma20']       = df['dollar_volume'].rolling(20).mean()
        df['dv_ratio']      = df['dollar_volume'] / (df['dv_ma20'] + 1e-9)

        # == MFI ==
        raw_mf   = tp * v
        mf_pos   = np.where(tp > tp.shift(1), raw_mf, 0.0)
        mf_neg   = np.where(tp < tp.shift(1), raw_mf, 0.0)
        mf_pos_s = pd.Series(mf_pos, index=df.index).rolling(14).sum()
        mf_neg_s = pd.Series(mf_neg, index=df.index).rolling(14).sum()
        df['mfi_14'] = 100 - 100 / (1 + mf_pos_s / (mf_neg_s + 1e-9))

        # == Keltner Channel ==
        kelt_mid = c.ewm(span=20, adjust=False).mean()
        kelt_atr = tr.rolling(10).mean()
        df['keltner_width']    = 4 * kelt_atr / (kelt_mid + 1e-9)
        df['keltner_position'] = (c - (kelt_mid - 2*kelt_atr)) / (4*kelt_atr + 1e-9)

        # == Heikin-Ashi ==
        ha_close = (o + h + l + c) / 4
        ha_open  = ((o.shift(1) + c.shift(1)) / 2).fillna(o)
        df['ha_candle_body'] = (ha_close - ha_open) / (c + 1e-9)
        ha_diff = ha_close.diff() > 0
        df['ha_trend']       = ha_diff.rolling(4).sum() / 4.0

        # == CMF ==
        mf_mult     = ((c - l) - (h - c)) / (h - l + 1e-9)
        df['cmf_20'] = (mf_mult * v).rolling(20).sum() / (v.rolling(20).sum() + 1e-9)

        # == Return Distribution Statistics ==
        df['return_skew_20']    = df['return'].rolling(20).skew()
        df['return_kurt_20']    = df['return'].rolling(20).kurt()
        df['return_autocorr_5'] = df['return'].rolling(20).corr(df['return'].shift(5)).fillna(0.0)

        # == ROC ==
        df['roc_10'] = (c - c.shift(10)) / (c.shift(10) + 1e-9) * 100
        df['roc_20'] = (c - c.shift(20)) / (c.shift(20) + 1e-9) * 100

        # == Ichimoku Cloud ==
        df['ichi_tenkan']  = (h.rolling(9).max()  + l.rolling(9).min())  / 2
        df['ichi_kijun']   = (h.rolling(26).max() + l.rolling(26).min()) / 2
        df['ichi_spana']   = ((df['ichi_tenkan'] + df['ichi_kijun']) / 2).shift(26)
        df['ichi_spanb']   = ((h.rolling(52).max() + l.rolling(52).min()) / 2).shift(26)
        cloud_top          = df[['ichi_spana','ichi_spanb']].max(axis=1)
        cloud_bot          = df[['ichi_spana','ichi_spanb']].min(axis=1)
        df['ichi_above']   = (c > cloud_top).astype(float)
        df['ichi_width']   = (cloud_top - cloud_bot) / (c + 1e-9)
        df['ichi_tk_cross'] = ((df['ichi_tenkan'] > df['ichi_kijun']) & 
                               (df['ichi_tenkan'].shift(1) <= df['ichi_kijun'].shift(1))).astype(float)

        # == Elder Ray Index ==
        ema13             = c.ewm(span=13, adjust=False).mean()
        df['elder_bull']  = (h - ema13) / (c + 1e-9)
        df['elder_bear']  = (l - ema13) / (c + 1e-9)

        # == Donchian Channel ==
        df['donchian_pos']   = (c - l.rolling(20).min()) / (h.rolling(20).max() - l.rolling(20).min() + 1e-9)
        df['donchian_width'] = (h.rolling(20).max() - l.rolling(20).min()) / (c + 1e-9)

        # == DPO ==
        shift_n       = 11
        df['dpo_20']  = (c.shift(shift_n) - c.rolling(20).mean().shift(shift_n)) / (c + 1e-9)

        # == Calendar ==
        if df['Date'].dtype != 'datetime64[ns]':
            df['Date'] = pd.to_datetime(df['Date'])
        dow = df['Date'].dt.dayofweek
        moy = df['Date'].dt.month
        df['day_sin']         = np.sin(2 * np.pi * dow / 5)
        df['day_cos']         = np.cos(2 * np.pi * dow / 5)
        df['month_sin']       = np.sin(2 * np.pi * moy / 12)
        df['month_cos']       = np.cos(2 * np.pi * moy / 12)
        df['is_january']      = (moy == 1).astype(float)
        df['is_december']     = (moy == 12).astype(float)
        df['is_month_end']    = df['Date'].dt.is_month_end.astype(float)
        df['is_quarter_end']  = df['Date'].dt.is_quarter_end.astype(float)

        # == Z-Scores ==
        for col_name in ['return', 'volatility_20', 'momentum_10', 'rel_volume', 'rsi_14']:
            if col_name in df.columns:
                df[f'{col_name}_z20'] = trailing_zscore(df[col_name], 20)

        # == Labels ==
        base_thr = max(CONFIG['target_return_threshold'], CONFIG['transaction_cost'] * 2)
        if CONFIG['adaptive_threshold']:
            vol_20     = df['return'].rolling(20).std()
            vol_med    = vol_20.rolling(60).median()
            adapt_sc   = (vol_20 / (vol_med + 1e-9)).clip(0.5, 2.0).fillna(1.0)
            move_thr   = base_thr * adapt_sc
        else:
            move_thr   = base_thr

        df['future_close']  = c.shift(-CONFIG['future_horizon'])
        df['future_return'] = df['future_close'] / c - 1
        df['target_up']     = (df['future_return'] > move_thr).astype(int)

        return df

    # ---- Get date splits ----
    def get_date_splits(dates, train_ratio=0.70, val_ratio=0.15):
        ud    = np.array(sorted(pd.to_datetime(pd.Series(dates)).unique()))
        t_end = int(len(ud) * train_ratio)
        v_end = int(len(ud) * (train_ratio + val_ratio))
        return ud[t_end], ud[v_end]

    train_cut_raw, _ = get_date_splits(raw_data['Date'].values)
    liquidity = (
        raw_data[raw_data['Date'] < train_cut_raw]
        .groupby('ticker', observed=True)['dollar_volume'].mean()
        .sort_values(ascending=False)
    )
    top_tickers = liquidity.head(CONFIG['top_n_tickers']).index.tolist()
    if 'SPY' in raw_data['ticker'].values and 'SPY' not in top_tickers:
        top_tickers.append('SPY')

    selected_raw = raw_data[raw_data['ticker'].isin(top_tickers)].copy()
    print(f'Processing features for {len(top_tickers)} tickers...')

    feat_dfs = []
    for ticker, grp in selected_raw.groupby('ticker', sort=False):
        if len(grp) < CONFIG['min_history_days']:
            continue
        try:
            feat_dfs.append(create_features(grp.reset_index(drop=True)))
        except Exception as e:
            print(f'  Feature creation failed for {ticker}: {e}')

    if not feat_dfs:
        raise RuntimeError('Feature engineering produced no data.')

    data = pd.concat(feat_dfs, ignore_index=True)
    data = data.sort_values(['ticker', 'Date']).reset_index(drop=True)
    print(f'After feature engineering: {data.shape}  |  tickers: {data["ticker"].nunique()}')


# =======================================================================
# Cell 3B - MARKET REGIME DETECTION (Volatility Clustering)
# =======================================================================

with cell_timer('Market Regime Detection'):

    def detect_volatility_regimes(spy_data, n_regimes=3):
        spy = spy_data.sort_values('Date').copy()
        spy['ret']           = spy['Close'].pct_change()
        spy['vol_5']         = spy['ret'].rolling(5).std()  * np.sqrt(252)
        spy['vol_20']        = spy['ret'].rolling(20).std() * np.sqrt(252)
        spy['vol_60']        = spy['ret'].rolling(60).std() * np.sqrt(252)
        spy['composite_vol'] = (0.3*spy['vol_5'] + 0.4*spy['vol_20'] + 0.3*spy['vol_60']).fillna(spy['vol_20'])
        spy['ma200']         = spy['Close'].rolling(200).mean()
        spy['trend']         = (spy['Close'] > spy['ma200']).astype(float).fillna(0.5)
        spy['vol_pct']       = spy['composite_vol'].rolling(252, min_periods=60).rank(pct=True)

        if n_regimes == 3:
            conds = [spy['vol_pct'] < 0.33, spy['vol_pct'] < 0.67]
            spy['regime'] = np.select(conds, [0, 1], default=2)
            spy['regime'] = spy['regime'].fillna(1).astype(int)
            spy.loc[(spy['regime'] == 0) & (spy['trend'] < 0.5), 'regime'] = 1
        else:
            spy['regime'] = pd.qcut(spy['vol_pct'].fillna(0.5), q=n_regimes, labels=False, duplicates='drop').fillna(1).astype(int)

        regime_names = {0: 'Low-Vol (Bull)', 1: 'Mid-Vol (Neutral)', 2: 'High-Vol (Bear)'}
        spy['regime_name'] = spy['regime'].map(regime_names)
        return spy[['Date', 'regime', 'regime_name', 'composite_vol', 'vol_pct', 'trend']].copy()

    if 'SPY' in data['ticker'].values:
        spy_raw   = (data[data['ticker']=='SPY'][['Date', 'Close']]
                     .drop_duplicates('Date').sort_values('Date'))
        regime_df = detect_volatility_regimes(spy_raw, n_regimes=CONFIG['n_regimes'])
        data = data.merge(regime_df[['Date', 'regime', 'regime_name', 'composite_vol']], on='Date', how='left')
        data['regime']        = data['regime'].fillna(1).astype(int)
        data['regime_name']   = data['regime_name'].fillna('Mid-Vol (Neutral)')
        data['composite_vol'] = data['composite_vol'].fillna(data['composite_vol'].median())
        regime_counts = data[data['ticker']!='SPY'].groupby('regime_name')['Date'].nunique()
        print('Regime distribution (unique trading days):')
        for rn, cnt in regime_counts.items():
            pct = cnt / regime_counts.sum() * 100
            print(f'  {rn:<22s}: {cnt:5d} days ({pct:.1f}%)')
    else:
        print('WARNING: SPY not found. Default regime = 1.')
        data['regime']        = 1
        data['regime_name']   = 'Unknown'
        data['composite_vol'] = 0.15

    # --- Regime visualization ---
    if 'SPY' in data['ticker'].values:
        spy_plt = data[data['ticker']=='SPY'].sort_values('Date').reset_index(drop=True)
        fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
        regime_bkg = {0: '#d4edda', 1: '#fff3cd', 2: '#f8d7da'}

        axes[0].plot(spy_plt['Date'], spy_plt['Close'], 'steelblue', lw=1.5, label='SPY Close')
        for rid, bkg in regime_bkg.items():
            mask = spy_plt['regime'] == rid
            if mask.any():
                axes[0].fill_between(spy_plt['Date'], spy_plt['Close'].min(), spy_plt['Close'].max(),
                                     where=mask, alpha=0.25, color=bkg, label=f'Regime {rid}')
        axes[0].set_ylabel('SPY Price ($)')
        axes[0].set_title('Market Regime Detection via Volatility Clustering', fontweight='bold')
        axes[0].legend(loc='upper left', fontsize=9)

        axes[1].plot(spy_plt['Date'], spy_plt['composite_vol']*100, 'darkorange', lw=1.5)
        axes[1].axhline(spy_plt['composite_vol'].quantile(0.33)*100, color='green', ls='--', alpha=0.7, label='33rd pct')
        axes[1].axhline(spy_plt['composite_vol'].quantile(0.67)*100, color='red',   ls='--', alpha=0.7, label='67th pct')
        axes[1].set_ylabel('Ann. Volatility (%)')
        axes[1].set_title('Composite Volatility Signal', fontweight='bold')
        axes[1].legend(fontsize=9)

        regime_colors_list = [regime_bkg.get(r, '#ffffff') for r in spy_plt['regime'].clip(0,2)]
        axes[2].bar(spy_plt['Date'], spy_plt['regime'], color=regime_colors_list, width=2)
        axes[2].set_yticks([0, 1, 2])
        axes[2].set_yticklabels(['Low-Vol', 'Mid-Vol', 'High-Vol'])
        axes[2].set_ylabel('Regime')
        axes[2].set_title('Regime Classification Timeline', fontweight='bold')
        axes[2].tick_params(axis='x', rotation=30)
        save_plot('regime_detection.png')
        print('Regime detection plot saved.')


# =======================================================================
# Cell 4 - MARKET & CROSS-SECTIONAL FEATURES (Leakage-Free)
# =======================================================================

with cell_timer('Market & Cross-Sectional Features'):

    def add_market_features(data):
        if 'SPY' not in data['ticker'].values:
            print('WARNING: SPY not found. Market features = 0.')
            for col in ['market_return', 'market_vol', 'market_regime',
                        'market_future_return', 'market_trend', 'market_skew', 'beta_20',
                        'idiosyncratic_return']:
                data[col] = 0
            return data

        spy = (data[data['ticker']=='SPY']
               [['Date', 'return', 'future_return', 'volatility_20', 'return_skew_20']]
               .drop_duplicates('Date').sort_values('Date')
               .rename(columns={'return':'market_return', 'future_return':'market_future_return',
                                'volatility_20':'market_vol', 'return_skew_20':'market_skew'}))
        spy['market_trend']  = spy['market_return'].rolling(50).mean()
        spy['market_regime'] = (spy['market_trend'] > 0).astype(int)

        merge_cols = ['Date', 'market_return', 'market_vol', 'market_regime',
                      'market_future_return', 'market_trend', 'market_skew']
        data = data.merge(spy[merge_cols], on='Date', how='left')
        for col in merge_cols[1:]:
            data[col] = data[col].fillna(0)
        data['market_regime'] = data['market_regime'].astype(int)

        def rolling_beta(grp, window=20):
            grp   = grp.sort_values('Date').copy()
            s_ret = grp['return']
            m_ret = grp['market_return']
            roll_cov = s_ret.rolling(window).cov(m_ret)
            roll_var = m_ret.rolling(window).var()
            betas    = (roll_cov / (roll_var + 1e-9)).to_numpy()
            grp['beta_20'] = np.clip(betas, -5, 5)
            return grp

        data = (data.groupby('ticker', sort=False, group_keys=False).apply(rolling_beta))
        data['beta_20']              = data['beta_20'].fillna(1.0)
        data['idiosyncratic_return'] = data['return'] - data['beta_20'] * data['market_return']
        return data

    def add_cross_sectional_features(data, date_col='Date', cols=None):
        if cols is None:
            cols = ['return', 'return_5d', 'momentum_10', 'rel_volume',
                    'volatility_20', 'rsi_14', 'ma_gap_20', 'macd_hist',
                    'bb_percent_b', 'idiosyncratic_return']
        data = data.copy()
        for col in cols:
            if col in data.columns:
                data[f'cs_rank_{col}'] = (
                    data.groupby(date_col, sort=False)[col].rank(pct=True, method='average')
                )
        return data

    data = add_market_features(data)
    cs_cols = ['return', 'return_5d', 'momentum_10', 'rel_volume', 'volatility_20',
               'rsi_14', 'ma_gap_20', 'macd_hist', 'bb_percent_b', 'idiosyncratic_return']
    data = add_cross_sectional_features(data, cols=cs_cols)

    print(f'After market/CS features: {data.shape}')
    print(f'New cols: beta_20, idiosyncratic_return, cs_rank_* ({len(cs_cols)} rank features)')


# =======================================================================
# Cell 5 - FEATURE SELECTION, TRAIN/VAL/TEST SPLIT & SCALING
# =======================================================================

with cell_timer('Feature Selection & Splitting'):

    all_feature_cols = [
        'price_diff','return','log_return','return_5d','return_10d','return_20d',
        'ma_gap_5','ma_gap_10','ma_gap_20','ma_gap_50','trend_strength','ma_cross',
        'volatility_5','volatility_10','volatility_20','vol_regime','park_vol',
        'momentum_5','momentum_10','momentum_20','momentum_60',
        'hl_range','oc_gap','intraday_return','close_position','candle_body',
        'upper_shadow','lower_shadow','doji',
        'return_lag_1','return_lag_2','return_lag_3','return_lag_5','return_lag_10',
        'volume_ratio_1','volume_ratio_3','volume_ratio_5','volume_ratio_10',
        'rsi_14','rsi_28','macd','macd_signal','macd_hist','macd_norm',
        'stoch_k','stoch_d','stoch_rsi',
        'bb_width','bb_percent_b','atr_14','atr_20',
        'obv_ratio','williams_r','cci_20','adx_14','plus_di','minus_di',
        'vwap_ratio','rel_volume','dv_ratio',
        'mfi_14','keltner_width','keltner_position',
        'ha_candle_body','ha_trend','cmf_20',
        'return_skew_20','return_kurt_20','return_autocorr_5',
        'roc_10','roc_20',
        'ichi_above','ichi_width','ichi_tk_cross',
        'elder_bull','elder_bear','donchian_pos','donchian_width','dpo_20',
        'return_z20','volatility_20_z20','momentum_10_z20','rel_volume_z20','rsi_14_z20',
        'day_sin','day_cos','month_sin','month_cos',
        'is_january','is_december','is_month_end','is_quarter_end',
        'market_return','market_vol','market_regime','market_trend','market_skew',
        'beta_20','idiosyncratic_return',
        'cs_rank_return','cs_rank_return_5d','cs_rank_momentum_10',
        'cs_rank_rel_volume','cs_rank_volatility_20','cs_rank_rsi_14',
        'cs_rank_ma_gap_20','cs_rank_macd_hist','cs_rank_bb_percent_b',
        'cs_rank_idiosyncratic_return',
        'regime','composite_vol',
    ]

    all_feature_cols = [c for c in all_feature_cols if c in data.columns]

    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=all_feature_cols + ['future_return', 'future_close', 'target_up'])
    data = data.sort_values(['ticker', 'Date']).reset_index(drop=True)

    train_cut, val_cut = get_date_splits(data['Date'].values)

    train_cut_ts = pd.Timestamp(train_cut)
    val_cut_ts   = pd.Timestamp(val_cut)
    embargo_td   = pd.Timedelta(days=CONFIG.get('embargo_days', 3))

    seq_pos    = data.groupby('ticker', sort=False).cumcount()
    valid_mask = seq_pos >= (CONFIG['seq_len'] - 1)

    train_mask = valid_mask & (data['Date'] <  train_cut_ts)
    val_mask   = valid_mask & (data['Date'] >= train_cut_ts + embargo_td) & (data['Date'] < val_cut_ts)
    test_mask  = valid_mask & (data['Date'] >= val_cut_ts + embargo_td)

    data['dataset_split'] = 'history'
    data.loc[train_mask, 'dataset_split'] = 'train'
    data.loc[val_mask,   'dataset_split'] = 'val'
    data.loc[test_mask,  'dataset_split'] = 'test'

    train_rows = data.loc[train_mask].copy().sort_values(['Date', 'ticker']).reset_index(drop=True)
    val_rows   = data.loc[val_mask  ].copy().sort_values(['Date', 'ticker']).reset_index(drop=True)
    test_rows  = data.loc[test_mask ].copy().sort_values(['Date', 'ticker']).reset_index(drop=True)

    X_train_full = train_rows[all_feature_cols].copy()
    y_train = train_rows['target_up'].astype(int)
    y_val   = val_rows['target_up'].astype(int)
    y_test  = test_rows['target_up'].astype(int)

    # --- Correlation pruning (fit only on train) ---
    non_const = [c for c in all_feature_cols if X_train_full[c].nunique(dropna=True) > 1]
    corr_mat  = X_train_full[non_const].corr().abs()
    upper     = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
    corr_drop = [c for c in upper.columns if any(upper[c] > CONFIG['feature_corr_threshold'])]
    candidate_features = [c for c in non_const if c not in corr_drop]

    # --- Mutual Information selection (fit only on train) ---
    mi_scores = mutual_info_classif(X_train_full[candidate_features], y_train, random_state=CONFIG['random_seed'])
    mi_df = (pd.DataFrame({'feature': candidate_features, 'mi_score': mi_scores})
             .sort_values('mi_score', ascending=False))
    positive_mi = mi_df[mi_df['mi_score'] > 0]
    if positive_mi.empty:
        positive_mi = mi_df

    max_feats    = min(CONFIG['max_features_modeling'], len(positive_mi))
    feature_cols = positive_mi.head(max_feats)['feature'].tolist()
    if not feature_cols:
        raise RuntimeError('Feature selection removed all features.')

    feat_report = mi_df.copy()
    feat_report['selected']            = feat_report['feature'].isin(feature_cols)
    feat_report['dropped_correlation'] = feat_report['feature'].isin(corr_drop)
    feat_report.to_csv(OUTPUT_DIRS['reports'] / 'feature_selection_report.csv', index=False)

    X_train = train_rows[feature_cols].copy()
    X_val   = val_rows[feature_cols].copy()
    X_test  = test_rows[feature_cols].copy()

    # --- Winsorization (fitted on train only) ---
    winsorize_pct    = CONFIG['winsorize_percentile']
    winsorize_bounds = {}
    for col in feature_cols:
        lo = float(X_train[col].quantile(winsorize_pct))
        hi = float(X_train[col].quantile(1 - winsorize_pct))
        winsorize_bounds[col] = (lo, hi)
        X_train[col] = X_train[col].clip(lo, hi)
        X_val[col]   = X_val[col].clip(lo, hi)
        X_test[col]  = X_test[col].clip(lo, hi)

    scaler = get_scaler(CONFIG['scaler_type'])
    scaler.fit(X_train)
    X_train_sc = scaler.transform(X_train)
    X_val_sc   = scaler.transform(X_val)
    X_test_sc  = scaler.transform(X_test)

    # --- DL sequences ---
    dl_scaled_frame = None
    if CONFIG['enable_dl_models']:
        data_win = data[feature_cols].copy()
        for col in feature_cols:
            lo, hi = winsorize_bounds[col]
            data_win[col] = data_win[col].clip(lo, hi)
        scaled_arr = scaler.transform(data_win)
        dl_scaled_frame = data[['Date', 'ticker', 'dataset_split', 'target_up']].copy()
        dl_scaled_frame[feature_cols] = scaled_arr

    # --- Time-decay sample weights ---
    def time_decay_weights(dates, min_w=0.35):
        ud  = pd.Series(sorted(pd.to_datetime(pd.Series(dates)).unique()))
        d2p = {d: i for i, d in enumerate(ud)}
        pos = pd.to_datetime(pd.Series(dates)).map(d2p).astype(float)
        sc  = pos / max(len(ud) - 1, 1)
        return (min_w + (1 - min_w) * sc).to_numpy(dtype=np.float32)

    train_weights = time_decay_weights(train_rows['Date'])
    pos_count     = float(y_train.sum())
    neg_count     = float(len(y_train) - pos_count)
    imb_ratio     = neg_count / (pos_count + 1e-6)

    split_summary = {
        'train_end':              str(train_cut_ts.date()),
        'val_start':              str((train_cut_ts + embargo_td).date()),
        'val_end':                str(val_cut_ts.date()),
        'test_start':             str((val_cut_ts + embargo_td).date()),
        'train_rows':             int(len(train_rows)),
        'val_rows':               int(len(val_rows)),
        'test_rows':              int(len(test_rows)),
        'raw_feature_count':      int(len(all_feature_cols)),
        'selected_feature_count': int(len(feature_cols)),
        'winsorization':          True,
        'data_hash':              DATA_HASH,
    }
    save_json_artifact(OUTPUT_DIRS['reports'] / 'data_split_summary.json', split_summary)

    print(f'Train end: {pd.Timestamp(train_cut).date()}  |  Val end: {pd.Timestamp(val_cut).date()}')
    print(f'Rows  -> Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
    print(f'Features: {len(feature_cols)} / {len(all_feature_cols)} selected')
    print(f'Positive rate -> Train: {y_train.mean():.4f}  Val: {y_val.mean():.4f}  Test: {y_test.mean():.4f}')
    print(f'Imbalance ratio: {imb_ratio:.2f}')


# =======================================================================
# Cell 6 - CLASSICAL MODELS (Logistic ElasticNet / LinearSVC / MLP)
# =======================================================================

with cell_timer('Classical Model Training'):
    from sklearn.linear_model import LogisticRegression
    from sklearn.neural_network import MLPClassifier
    from sklearn.svm import LinearSVC

    model_registry = {}
    X_train_model = pd.DataFrame(X_train_sc, columns=feature_cols, index=X_train.index)
    X_val_model   = pd.DataFrame(X_val_sc,   columns=feature_cols, index=X_val.index)
    X_test_model  = pd.DataFrame(X_test_sc,  columns=feature_cols, index=X_test.index)

    def safe_pr_auc(y_true, probs):
        try:
            return float(average_precision_score(y_true, probs))
        except Exception:
            return float('nan')

    def safe_fit(model, X_fit, y_fit, sw=None):
        try:
            if sw is not None:
                model.fit(X_fit, y_fit, sample_weight=sw)
            else:
                model.fit(X_fit, y_fit)
        except TypeError:
            model.fit(X_fit, y_fit)
        return model

    def squash_to_prob(raw):
        raw = np.asarray(raw, dtype=float)
        if raw.size == 0:
            return raw
        if np.nanmin(raw) < 0 or np.nanmax(raw) > 1:
            raw = expit(raw)
        return np.clip(raw, 1e-6, 1 - 1e-6)

    def get_raw_scores(model, X_df, score_kind):
        if score_kind == 'decision_function':
            return np.asarray(model.decision_function(X_df), dtype=float)
        if score_kind == 'predict_proba':
            return np.asarray(model.predict_proba(X_df)[:, 1], dtype=float)
        raise ValueError(f'Unknown score_kind: {score_kind}')

    def derive_fi(model):
        if hasattr(model, 'coef_'):
            return np.abs(np.asarray(model.coef_)).reshape(-1)
        if hasattr(model, 'coefs_') and model.coefs_:
            return np.mean(np.abs(np.asarray(model.coefs_[0])), axis=1)
        return None

    def register_classical(name, model, score_kind, meta):
        tr_raw  = get_raw_scores(model, X_train_model, score_kind)
        vl_raw  = get_raw_scores(model, X_val_model,   score_kind)
        te_raw  = get_raw_scores(model, X_test_model,  score_kind)
        tr_prob = squash_to_prob(tr_raw)
        vl_prob = squash_to_prob(vl_raw)
        te_prob = squash_to_prob(te_raw)
        val_ic  = information_coefficient(vl_prob, val_rows['future_return'].values)
        model_registry[name] = {
            'model': model, 'type': 'classical', 'score_kind': score_kind,
            'feature_space': 'scaled',
            'train_probs': tr_prob, 'val_probs': vl_prob, 'test_probs': te_prob,
            'train_raw_scores': tr_raw, 'val_raw_scores': vl_raw, 'test_raw_scores': te_raw,
            'val_auc':    float(roc_auc_score(y_val, vl_raw if score_kind=='decision_function' else vl_prob)),
            'val_pr_auc': safe_pr_auc(y_val, vl_prob),
            'val_ic':     val_ic,
            'feature_importance': derive_fi(model),
            'meta': meta,
        }

    search_log = []

    # ---- Logistic ElasticNet -----------------------------------------
    print('\n' + '='*60)
    print('Training Logistic ElasticNet...')
    print('='*60)
    best_score, best_model, best_meta = -np.inf, None, None
    for C in [0.05, 0.10, 0.50, 1.00, 2.00]:
        for l1_ratio in [0.0, 0.2, 0.5, 0.8]:
            m = LogisticRegression(
                penalty='elasticnet', solver='saga',
                max_iter=4000, class_weight='balanced',
                random_state=CONFIG['random_seed'],
                C=C, l1_ratio=l1_ratio
            )
            safe_fit(m, X_train_model, y_train, sw=train_weights)
            vp  = m.predict_proba(X_val_model)[:, 1]
            auc = roc_auc_score(y_val, vp)
            bri = brier_score_loss(y_val, vp)
            ic  = information_coefficient(vp, val_rows['future_return'].values)
            scr = 0.6*auc + 0.3*ic - 0.1*bri
            search_log.append({
                'model':'LogisticElasticNet', 'C':C, 'l1_ratio':l1_ratio,
                'val_auc':auc, 'val_ic':ic, 'proxy_score':scr
            })
            if scr > best_score:
                best_score, best_model = scr, m
                best_meta = {'C':C, 'l1_ratio':l1_ratio, 'proxy_score':scr}
    register_classical('LogisticElasticNet', best_model, 'predict_proba', best_meta)
    print(f'  Val AUC: {model_registry["LogisticElasticNet"]["val_auc"]:.4f}  '
          f'IC: {model_registry["LogisticElasticNet"]["val_ic"]:.4f}')

    # ---- LinearSVC -------------------------------------------------------
    print('\n' + '='*60)
    print('Training LinearSVC...')
    print('='*60)
    best_score, best_model, best_meta = -np.inf, None, None
    for C in [0.01, 0.05, 0.10, 0.50, 1.00, 2.00]:
        m = LinearSVC(
            C=C, class_weight='balanced', max_iter=8000,
            random_state=CONFIG['random_seed']
        )
        safe_fit(m, X_train_model, y_train, sw=train_weights)
        vr  = m.decision_function(X_val_model)
        vp  = squash_to_prob(vr)
        auc = roc_auc_score(y_val, vr)
        bri = brier_score_loss(y_val, vp)
        ic  = information_coefficient(vp, val_rows['future_return'].values)
        scr = 0.6*auc + 0.3*ic - 0.1*bri
        search_log.append({'model':'LinearSVC', 'C':C, 'val_auc':auc, 'val_ic':ic, 'proxy_score':scr})
        if scr > best_score:
            best_score, best_model = scr, m
            best_meta = {'C':C, 'proxy_score':scr}
    register_classical('LinearSVC', best_model, 'decision_function', best_meta)
    print(f'  Val AUC: {model_registry["LinearSVC"]["val_auc"]:.4f}  '
          f'IC: {model_registry["LinearSVC"]["val_ic"]:.4f}')

    # ---- MLP Classifier --------------------------------------------------
    print('\n' + '='*60)
    print('Training MLP Classifier...')
    print('='*60)
    best_score, best_model, best_meta = -np.inf, None, None
    mlp_grid = [
        {'hs':(64,),          'alpha':1e-4, 'lr':1e-3},
        {'hs':(128,),         'alpha':5e-4, 'lr':1e-3},
        {'hs':(128, 64),      'alpha':1e-3, 'lr':5e-4},
        {'hs':(256, 128, 64), 'alpha':1e-3, 'lr':3e-4},
    ]
    bs = min(CONFIG['batch_size'], max(len(X_train_model) // 8, 32))
    for cfg in mlp_grid:
        m = MLPClassifier(
            hidden_layer_sizes=cfg['hs'], activation='relu',
            solver='adam', alpha=cfg['alpha'],
            batch_size=bs, learning_rate_init=cfg['lr'],
            max_iter=300, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=20,
            random_state=CONFIG['random_seed']
        )
        safe_fit(m, X_train_model, y_train)
        vp  = m.predict_proba(X_val_model)[:, 1]
        auc = roc_auc_score(y_val, vp)
        bri = brier_score_loss(y_val, vp)
        ic  = information_coefficient(vp, val_rows['future_return'].values)
        scr = 0.6*auc + 0.3*ic - 0.1*bri
        search_log.append({
            'model':'MLPClassifier', 'hidden':cfg['hs'], 'alpha':cfg['alpha'],
            'val_auc':auc, 'val_ic':ic, 'proxy_score':scr
        })
        if scr > best_score:
            best_score, best_model = scr, m
            best_meta = {'hidden_layer_sizes':cfg['hs'], 'alpha':cfg['alpha'], 'lr':cfg['lr'], 'proxy_score':scr}
    register_classical('MLPClassifier', best_model, 'predict_proba', best_meta)
    print(f'  Val AUC: {model_registry["MLPClassifier"]["val_auc"]:.4f}  '
          f'IC: {model_registry["MLPClassifier"]["val_ic"]:.4f}')

    save_json_artifact(OUTPUT_DIRS['reports'] / 'classical_model_search.json', search_log)
    print('\n--- Classical Models Summary ---')
    for name, info in model_registry.items():
        print(f'  {name:<22s} Val AUC: {info["val_auc"]:.4f}  '
              f'Val PR-AUC: {info["val_pr_auc"]:.4f}  Val IC: {info["val_ic"]:.4f}')


# =======================================================================
# Cell 7 - DEEP LEARNING MODELS
#          Attention-LSTM | Dilated Causal CNN | Temporal Transformer
# =======================================================================

with cell_timer('Deep Learning Models'):

    class SequenceDataset(Dataset):
        def __init__(self, x, y):
            self.X = torch.tensor(x, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.float32)
        def __len__(self):  return len(self.y)
        def __getitem__(self, i): return self.X[i], self.y[i]

    def make_loader(ds, bs, shuffle):
        nw = 2 if CONFIG['device'] == 'cuda' else 0
        return DataLoader(
            ds, batch_size=bs, shuffle=shuffle,
            num_workers=nw, pin_memory=(CONFIG['device']=='cuda'),
            persistent_workers=(nw > 0)
        )

    def build_sequences(frame, selected_cols, seq_len):
        split_rank = {'train':0, 'val':1, 'test':2}
        store = {s: {'X':[], 'y':[], 'meta':[]} for s in ['train', 'val', 'test']}
        for ticker, grp in frame.groupby('ticker', sort=False):
            grp = grp.sort_values('Date').reset_index(drop=True)
            feat   = grp[selected_cols].to_numpy(dtype=np.float32)
            target = grp['target_up'].to_numpy(dtype=np.float32)
            splits = grp['dataset_split'].tolist()
            dates  = grp['Date'].tolist()
            for end in range(seq_len-1, len(grp)):
                cur_sp = splits[end]
                if cur_sp not in store:
                    continue
                win_sp = splits[end-seq_len+1 : end+1]
                max_r  = split_rank[cur_sp]
                if any(split_rank.get(s,-1) > max_r for s in win_sp if s in split_rank):
                    continue
                store[cur_sp]['X'].append(feat[end-seq_len+1 : end+1])
                store[cur_sp]['y'].append(target[end])
                store[cur_sp]['meta'].append({'ticker':ticker, 'date':str(pd.Timestamp(dates[end]).date())})
        
        def finalize(sp):
            xs = store[sp]['X']; ys = store[sp]['y']
            xa = np.asarray(xs, dtype=np.float32) if xs else np.empty((0, seq_len, len(selected_cols)), dtype=np.float32)
            ya = np.asarray(ys, dtype=np.float32) if ys else np.empty((0,), dtype=np.float32)
            return xa, ya, pd.DataFrame(store[sp]['meta'])
        
        return {s: finalize(s) for s in ['train', 'val', 'test']}

    def predict_dl(model, loader):
        model.eval()
        out, amp = [], CONFIG['device'] == 'cuda'
        with torch.no_grad():
            for xb, _ in loader:
                xb = xb.to(CONFIG['device'], non_blocking=True)
                with torch.amp.autocast(device_type='cuda', enabled=amp):
                    logits = model(xb)
                p = torch.sigmoid(logits).cpu().numpy()
                out.extend(p.tolist())
        return np.array(out)

    # -------------------------------------------------------------------
    # Model 1: Attention-LSTM
    # -------------------------------------------------------------------
    class AttentionLSTM(nn.Module):
        def __init__(self, input_size, hidden=128, layers=2, dropout=0.3, n_heads=4):
            super().__init__()
            self.lstm   = nn.LSTM(input_size, hidden, layers, batch_first=True, dropout=dropout)
            self.attn   = nn.MultiheadAttention(hidden, n_heads, dropout=dropout*0.5, batch_first=True)
            self.norm1  = nn.LayerNorm(hidden)
            self.norm2  = nn.LayerNorm(hidden)
            self.head   = nn.Sequential(
                nn.Linear(hidden, 64), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(64, 1)
            )

        def forward(self, x):
            out, _ = self.lstm(x)
            attn_out, _ = self.attn(out, out, out)
            out = self.norm1(out + attn_out)           # residual
            pooled  = out.mean(dim=1)                  # global average pool
            last    = out[:, -1, :]                    # last timestep
            fused   = self.norm2(pooled + last)        # combine
            return self.head(fused).squeeze(-1)

    # -------------------------------------------------------------------
    # Model 2: Dilated Causal CNN (TCN-style)
    # -------------------------------------------------------------------
    class DilatedCausalConv(nn.Module):
        def __init__(self, in_ch, out_ch, kernel=3, dilation=1, dropout=0.2):
            super().__init__()
            pad = (kernel - 1) * dilation
            self.conv = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation, padding=pad)
            self.trim  = pad
            self.norm  = nn.BatchNorm1d(out_ch)
            self.act   = nn.GELU()
            self.drop  = nn.Dropout(dropout)
            self.res   = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

        def forward(self, x):
            out = self.conv(x)
            if self.trim > 0:
                out = out[:, :, :-self.trim]   # causal: remove future context
            out = self.norm(out)
            out = self.drop(self.act(out))
            res = self.res(x) if self.res else x
            return out + res

    class DilatedCausalCNN(nn.Module):
        def __init__(self, input_size, seq_len, ch=64, dropout=0.3):
            super().__init__()
            self.input_proj = nn.Conv1d(input_size, ch, 1)
            dilations = [1, 2, 4, 8, 16]
            self.blocks = nn.ModuleList([
                DilatedCausalConv(ch, ch, kernel=3, dilation=d, dropout=dropout)
                for d in dilations
            ])
            self.head = nn.Sequential(
                nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                nn.Linear(ch, 32), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            x = x.permute(0, 2, 1)          # (B, features, T)
            x = self.input_proj(x)
            for block in self.blocks:
                x = block(x)
            return self.head(x).squeeze(-1)

    # -------------------------------------------------------------------
    # Model 3: Temporal Transformer (NEW)
    # -------------------------------------------------------------------
    class TemporalTransformer(nn.Module):
        """
        Decoder-only Transformer for time-series classification.
        Uses causal (masked) self-attention to prevent look-ahead leakage.
        """
        def __init__(self, input_size, seq_len, d_model=64, n_heads=4, n_layers=2, dropout=0.2):
            super().__init__()
            self.input_proj = nn.Linear(input_size, d_model)
            # Learnable positional embedding
            self.pos_emb    = nn.Embedding(seq_len, d_model)
            encoder_layer   = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                dropout=dropout, activation='gelu', batch_first=True,
                norm_first=True
            )         # Pre-LN (more stable training)
            self.encoder    = nn.TransformerEncoder(encoder_layer, n_layers)
            self.head       = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, 32), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(32, 1)
            )
            self.seq_len    = seq_len

        def _causal_mask(self, sz, device):
            mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
            return mask                    # True = ignore (attend only to past)

        def forward(self, x):
            B, T, _ = x.shape
            pos  = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
            x    = self.input_proj(x) + self.pos_emb(pos)
            mask = self._causal_mask(T, x.device)
            out  = self.encoder(x, mask=mask)
            return self.head(out[:, -1, :]).squeeze(-1)   # use last timestep

    # -------------------------------------------------------------------
    # Generic DL Training Loop
    # -------------------------------------------------------------------
    def train_dl_model(model, train_ldr, val_ldr, epochs, name):
        model = model.to(CONFIG['device'])
        opt   = torch.optim.AdamW(
            model.parameters(),
            lr=CONFIG['dl_lr'],
            weight_decay=CONFIG['dl_weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
        ls        = CONFIG['label_smoothing']
        amp_on    = CONFIG['device'] == 'cuda'
        scaler_amp = torch.amp.GradScaler('cuda', enabled=amp_on)
        patience   = CONFIG['dl_patience']
        best_val   = np.inf
        best_state = None
        no_improve = 0
        history    = {'train_loss':[], 'val_loss':[], 'val_auc':[], 'lr':[]}

        for epoch in range(epochs):
            # ---- Train ----
            model.train()
            train_losses = []
            for xb, yb in train_ldr:
                xb = xb.to(CONFIG['device'], non_blocking=True)
                yb = yb.to(CONFIG['device'], non_blocking=True)
                yb_smooth = yb * (1 - ls) + 0.5 * ls   # label smoothing
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type='cuda', enabled=amp_on):
                    logits = model(xb)
                    loss   = F.binary_cross_entropy_with_logits(logits, yb_smooth)
                scaler_amp.scale(loss).backward()
                scaler_amp.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['dl_grad_clip'])
                scaler_amp.step(opt)
                scaler_amp.update()
                train_losses.append(loss.item())

            # ---- Validate ----
            model.eval()
            val_losses, val_preds, val_labels = [], [], []
            with torch.no_grad():
                for xb, yb in val_ldr:
                    xb = xb.to(CONFIG['device'], non_blocking=True)
                    yb = yb.to(CONFIG['device'], non_blocking=True)
                    with torch.amp.autocast(device_type='cuda', enabled=amp_on):
                        logits = model(xb)
                        loss   = F.binary_cross_entropy_with_logits(logits, yb)
                    val_losses.append(loss.item())
                    val_preds.extend(torch.sigmoid(logits).cpu().numpy().tolist())
                    val_labels.extend(yb.cpu().numpy().tolist())  

            scheduler.step()

            tl = np.mean(train_losses)
            vl = np.mean(val_losses)
            try:
                va = roc_auc_score(val_labels, val_preds)  
            except Exception:
                va = 0.5

            history['train_loss'].append(tl)
            history['val_loss'].append(vl)
            history['val_auc'].append(va)
            history['lr'].append(scheduler.get_last_lr()[0])

            if (epoch + 1) % 5 == 0:
                print(f'  [{name}] Epoch {epoch+1:3d}/{epochs} | '
                      f'Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | '
                      f'Val AUC: {va:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')

            # Early stopping
            if vl < best_val - 1e-5:
                best_val   = vl
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f'  [{name}] Early stopping at epoch {epoch+1} (best val loss: {best_val:.4f})')
                    break

        if best_state:
            model.load_state_dict(best_state)
        return model, history

    def register_dl_model(name, model, train_ldr, val_ldr, test_ldr, history, val_meta, test_meta, y_val_dl, y_test_dl):
        val_probs  = predict_dl(model, val_ldr)
        test_probs = predict_dl(model, test_ldr)
        train_probs = predict_dl(model, train_ldr)

        val_ret_aligned  = val_meta.merge(
            val_rows[['ticker', 'Date', 'future_return']].assign(date=val_rows['Date'].dt.strftime('%Y-%m-%d')),
            left_on=['ticker', 'date'], right_on=['ticker', 'date'], how='left'
        )['future_return'].fillna(0).values
        val_ic = information_coefficient(val_probs, val_ret_aligned[:len(val_probs)])

        try:
            va  = float(roc_auc_score(y_val_dl, val_probs))
        except Exception:
            va  = 0.5

        model_registry[name] = {
            'model': model, 'type': 'dl',
            'train_probs': train_probs,
            'val_probs':   val_probs,
            'test_probs':  test_probs,
            'val_auc':     va,
            'val_pr_auc':  safe_pr_auc(y_val_dl, val_probs),
            'val_ic':      val_ic,
            'history':     history,
            'val_meta':    val_meta,
            'test_meta':   test_meta,
            'feature_importance': None,
            'meta': {'epochs_run': len(history['train_loss'])},
        }
        print(f'  {name:<22s} Val AUC: {va:.4f}  Val IC: {val_ic:.4f}')

    if CONFIG['enable_dl_models'] and dl_scaled_frame is not None:
        print('\nBuilding sequence datasets...')
        seqs = build_sequences(dl_scaled_frame, feature_cols, CONFIG['seq_len'])

        X_tr_seq, y_tr_seq, _       = seqs['train']
        X_va_seq, y_va_seq, va_meta = seqs['val']
        X_te_seq, y_te_seq, te_meta = seqs['test']
        print(f'  Sequences -> Train: {len(y_tr_seq)}  Val: {len(y_va_seq)}  Test: {len(y_te_seq)}')

        bs  = CONFIG['batch_size']
        tr_ds = SequenceDataset(X_tr_seq, y_tr_seq)
        va_ds = SequenceDataset(X_va_seq, y_va_seq)
        te_ds = SequenceDataset(X_te_seq, y_te_seq)
        tr_ldr = make_loader(tr_ds, bs, shuffle=True)
        va_ldr = make_loader(va_ds, bs, shuffle=False)
        te_ldr = make_loader(te_ds, bs, shuffle=False)

        input_sz = len(feature_cols)

        # ---- AttentionLSTM ----
        print('\n' + '='*60)
        print('Training Attention-LSTM...')
        lstm_model = AttentionLSTM(input_sz, hidden=128, layers=2, dropout=0.3, n_heads=4)
        lstm_model, lstm_hist = train_dl_model(lstm_model, tr_ldr, va_ldr, CONFIG['lstm_epochs'], 'LSTM')
        register_dl_model('AttentionLSTM', lstm_model, tr_ldr, va_ldr, te_ldr, lstm_hist, va_meta, te_meta, y_va_seq, y_te_seq)

        # ---- DilatedCausalCNN ----
        print('\n' + '='*60)
        print('Training Dilated Causal CNN...')
        cnn_model = DilatedCausalCNN(input_sz, CONFIG['seq_len'], ch=64, dropout=0.3)
        cnn_model, cnn_hist = train_dl_model(cnn_model, tr_ldr, va_ldr, CONFIG['tcn_epochs'], 'CNN')
        register_dl_model('DilatedCausalCNN', cnn_model, tr_ldr, va_ldr, te_ldr, cnn_hist, va_meta, te_meta, y_va_seq, y_te_seq)

        # ---- Temporal Transformer ----
        print('\n' + '='*60)
        print('Training Temporal Transformer (NEW)...')
        tfm_model = TemporalTransformer(input_sz, CONFIG['seq_len'], d_model=64, n_heads=4, n_layers=2, dropout=0.2)
        tfm_model, tfm_hist = train_dl_model(tfm_model, tr_ldr, va_ldr, CONFIG['transformer_epochs'], 'Transformer')
        register_dl_model('TemporalTransformer', tfm_model, tr_ldr, va_ldr, te_ldr, tfm_hist, va_meta, te_meta, y_va_seq, y_te_seq)

        # ---- DL Training Curves Plot ----
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        dl_models_plot = [
            ('AttentionLSTM',       lstm_hist,  PAPER_COLORS[0]),
            ('DilatedCausalCNN',    cnn_hist,   PAPER_COLORS[1]),
            ('TemporalTransformer', tfm_hist,   PAPER_COLORS[2]),
        ]
        for ax, (mname, hist, color) in zip(axes, dl_models_plot):
            ep = range(1, len(hist['train_loss'])+1)
            ax.plot(ep, hist['train_loss'], '-',  color=color,  lw=2, label='Train Loss')
            ax.plot(ep, hist['val_loss'],   '--', color=color,  lw=2, alpha=0.8, label='Val Loss')
            ax2 = ax.twinx()
            ax2.plot(ep, hist['val_auc'], ':', color='darkred', lw=1.5, label='Val AUC')
            ax2.set_ylabel('Val AUC', color='darkred')
            ax2.tick_params(axis='y', colors='darkred')
            ax2.set_ylim(0.4, 1.0)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.set_title(f'{mname}\nTraining Dynamics', fontweight='bold')
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper right')
        save_plot('dl_training_curves.png')
        print('\nDL training curves saved.')

    else:
        print('DL models disabled or dl_scaled_frame not available.')

    print('\n--- All Models Summary ---')
    for name, info in sorted(model_registry.items(), key=lambda x: -x[1].get('val_auc', 0)):
        print(f'  {name:<22s} Val AUC: {info["val_auc"]:.4f}  '
              f'Val IC: {info.get("val_ic", 0):.4f}  Type: {info["type"]}')


# =======================================================================
# Cell 8 - WALK-FORWARD (ROLLING) VALIDATION
# =======================================================================

with cell_timer('Walk-Forward Validation'):
    from sklearn.linear_model import LogisticRegression

    wf_results = []
    n_folds    = CONFIG['wf_n_folds']
    min_train  = CONFIG['wf_min_train_days']

    all_dates  = np.array(sorted(data['Date'].unique()))
    n_dates    = len(all_dates)

    # Fold boundaries: expanding train window + fixed val window
    fold_size  = (n_dates - min_train) // n_folds
    if fold_size < 20:
        print(f'Not enough data for {n_folds} folds. Skipping walk-forward.')
        wf_results = []
    else:
        print(f'Walk-Forward: {n_folds} folds | fold_size={fold_size} days')
        print('='*68)

        for fold_idx in range(n_folds):
            train_end_idx = min_train + fold_idx * fold_size
            val_end_idx   = min(train_end_idx + fold_size, n_dates - 1)

            train_end_date = all_dates[train_end_idx]
            val_end_date   = all_dates[val_end_idx]

            # Embargo: skip CONFIG['embargo_days'] days after train end
            embargo_idx    = min(train_end_idx + CONFIG['embargo_days'], n_dates-1)
            val_start_date = all_dates[embargo_idx]

            fold_train_mask = data['Date'] <  train_end_date
            fold_val_mask   = ((data['Date'] >= val_start_date) & (data['Date'] <  val_end_date))

            fold_train = data[fold_train_mask & (data['dataset_split'] != 'test')].copy()
            fold_val   = data[fold_val_mask].copy()

            if len(fold_train) < 500 or len(fold_val) < 50:
                print(f'  Fold {fold_idx+1}: insufficient data, skipping.')
                continue

            # Drop NaN features
            fold_train = fold_train.dropna(subset=feature_cols + ['target_up']).reset_index(drop=True)
            fold_val   = fold_val.dropna(subset=feature_cols + ['target_up']).reset_index(drop=True)

            X_f_tr = fold_train[feature_cols].copy()
            y_f_tr = fold_train['target_up'].astype(int)
            X_f_va = fold_val[feature_cols].copy()
            y_f_va = fold_val['target_up'].astype(int)

            if y_f_va.nunique() < 2:
                print(f'  Fold {fold_idx+1}: single-class val, skipping.')
                continue

            # Winsorize + scale (fit on fold train only)
            wf_bounds = {}
            for col in feature_cols:
                lo = float(X_f_tr[col].quantile(winsorize_pct))
                hi = float(X_f_tr[col].quantile(1 - winsorize_pct))
                wf_bounds[col] = (lo, hi)
                X_f_tr[col]    = X_f_tr[col].clip(lo, hi)
                X_f_va[col]    = X_f_va[col].clip(lo, hi)

            wf_scaler = get_scaler(CONFIG['scaler_type'])
            wf_scaler.fit(X_f_tr)
            X_f_tr_sc = wf_scaler.transform(X_f_tr)
            X_f_va_sc = wf_scaler.transform(X_f_va)

            X_f_tr_df = pd.DataFrame(X_f_tr_sc, columns=feature_cols)
            X_f_va_df = pd.DataFrame(X_f_va_sc, columns=feature_cols)

            # Use a fast-fitting Logistic model for walk-forward
            wf_model = LogisticRegression(
                penalty='elasticnet', solver='saga', max_iter=3000,
                class_weight='balanced', C=0.5, l1_ratio=0.3,
                random_state=CONFIG['random_seed']
            )
            wf_model.fit(X_f_tr_df, y_f_tr)
            wf_probs = wf_model.predict_proba(X_f_va_df)[:, 1]
            wf_preds = (wf_probs >= 0.5).astype(int)

            fold_auc = float(roc_auc_score(y_f_va, wf_probs))
            fold_f1  = float(f1_score(y_f_va, wf_preds, zero_division=0))
            fold_mcc = float(matthews_corrcoef(y_f_va, wf_preds))
            fold_ic  = information_coefficient(wf_probs, fold_val['future_return'].fillna(0).values)

            # Fold backtest: long top-30% by probability
            fold_bt = fold_val.copy()
            fold_bt['prob'] = wf_probs
            fold_bt_daily = []
            for date, grp in fold_bt.groupby('Date', sort=True):
                n_long    = max(1, int(len(grp) * CONFIG['long_only_top_pct']))
                long_idx  = grp['prob'].nlargest(n_long).index
                grp_rets  = fold_val.loc[long_idx, 'future_return'].fillna(0).mean()
                fold_bt_daily.append({'date': date, 'ret': grp_rets})
            
            fold_bt_df = pd.DataFrame(fold_bt_daily)
            if len(fold_bt_df) > 1:
                fold_ret    = fold_bt_df['ret'].values
                fold_sharpe = (fold_ret.mean() / (fold_ret.std() + 1e-9)) * np.sqrt(252)
                fold_cum    = float((1 + fold_bt_df['ret']).prod() - 1)
            else:
                fold_sharpe = 0.0
                fold_cum    = 0.0

            result = {
                'fold':              fold_idx + 1,
                'train_start':       str(all_dates[0].date()),
                'train_end':         str(train_end_date.date()),
                'val_start':         str(val_start_date.date()),
                'val_end':           str(val_end_date.date()),
                'train_rows':        int(len(fold_train)),
                'val_rows':          int(len(fold_val)),
                'val_auc':           fold_auc,
                'val_f1':            fold_f1,
                'val_mcc':           fold_mcc,
                'val_ic':            fold_ic,
                'val_sharpe':        fold_sharpe,
                'val_cum_return':    fold_cum,
                'val_positive_rate': float(y_f_va.mean()),
            }
            wf_results.append(result)
            print(f'  Fold {fold_idx+1}/5 | '
                  f'AUC: {fold_auc:.4f} | F1: {fold_f1:.4f} | '
                  f'IC: {fold_ic:.4f} | Sharpe: {fold_sharpe:.2f} | '
                  f'CumRet: {fold_cum:.2%}')

        if wf_results:
            wf_df = pd.DataFrame(wf_results)
            wf_df.to_csv(OUTPUT_DIRS['walkforward'] / 'walk_forward_results.csv', index=False)

            print('\n' + '='*68)
            print('Walk-Forward Summary:')
            metrics_to_summarize = ['val_auc', 'val_f1', 'val_mcc', 'val_ic', 'val_sharpe', 'val_cum_return']
            for m in metrics_to_summarize:
                vals = wf_df[m].dropna()
                print(f'  {m:<22s}  mean={vals.mean():.4f}  '
                      f'std={vals.std():.4f}  '
                      f'min={vals.min():.4f}  max={vals.max():.4f}')

            # IC Information Ratio
            ic_vals = wf_df['val_ic'].dropna().values
            ir      = information_ratio(ic_vals)
            print(f'\n  Information Ratio (IC/IC_std): {ir:.4f}')
            print(f'  Stable signal: {"YES" if ir > 0.5 else "MARGINAL" if ir > 0.3 else "UNSTABLE"}')

            # Walk-Forward Visualization
            fig, axes = plt.subplots(2, 3, figsize=(18, 10))
            metric_pairs = [
                ('val_auc',        'ROC-AUC',           PAPER_COLORS[0]),
                ('val_f1',         'F1 Score',          PAPER_COLORS[1]),
                ('val_mcc',        'MCC',               PAPER_COLORS[2]),
                ('val_ic',         'IC (Spearman)',     PAPER_COLORS[3]),
                ('val_sharpe',     'Sharpe Ratio',      PAPER_COLORS[4]),
                ('val_cum_return', 'Cumulative Return', PAPER_COLORS[5]),
            ]
            folds = wf_df['fold'].values
            for ax, (col, label, color) in zip(axes.flat, metric_pairs):
                vals = wf_df[col].values
                bars = ax.bar(folds, vals, color=color, edgecolor='white', alpha=0.85, width=0.6)
                ax.axhline(vals.mean(), color='black', ls='--', lw=2, label=f'Mean={vals.mean():.4f}')
                ax.axhline(0, color='gray', ls=':', alpha=0.4)
                for bar, v in zip(bars, vals):
                    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height() + abs(vals.max()-vals.min())*0.02,
                            f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
                ax.set_xlabel('Fold')
                ax.set_ylabel(label)
                ax.set_title(f'Walk-Forward: {label}', fontweight='bold')
                ax.set_xticks(folds)
                ax.legend(fontsize=9)
            save_plot('walk_forward_validation.png')
            print('Walk-forward plot saved.')
        else:
            print('No walk-forward results produced.')


# =======================================================================
# Cell 9 - BEST MODEL SELECTION + CALIBRATION
# =======================================================================

with cell_timer('Best Model Selection'):
    from sklearn.isotonic import IsotonicRegression

    # ---- Isotonic Calibration (Applied Upfront) ----------------------
    for name, info in model_registry.items():
        try:
            vp_cal = info['val_probs'][:len(y_val)]
            iso    = IsotonicRegression(out_of_bounds='clip')
            iso.fit(vp_cal, y_val[:len(vp_cal)])

            # Calibrate validation and test probabilities
            info['val_probs'] = np.clip(iso.predict(info['val_probs']), 1e-6, 1-1e-6)
            info['test_probs'] = np.clip(iso.predict(info['test_probs']), 1e-6, 1-1e-6)
            info['calibrator'] = iso
            info['cal_test_probs'] = info['test_probs'] # Backwards compatible
        except Exception as e:
            pass

    def sharpe_from_probs(probs, future_returns, threshold=0.5, top_pct=0.30):
        probs          = np.asarray(probs, dtype=float)
        future_returns = np.asarray(future_returns, dtype=float)
        n_select   = max(1, int(len(probs) * top_pct))
        top_idx    = np.argsort(probs)[-n_select:]
        daily_rets = future_returns[top_idx]
        valid      = daily_rets[~np.isnan(daily_rets)]
        if len(valid) < 2:
            return 0.0
        return float((valid.mean() / (valid.std() + 1e-9)) * np.sqrt(252))

    def composite_selection_score(info, y_val_arr, val_future_rets):
        w       = CONFIG['selection_weights']
        va      = info.get('val_auc', 0.5)
        pr      = info.get('val_pr_auc', 0.0) or 0.0
        ic      = info.get('val_ic', 0.0) or 0.0
        sharpe  = sharpe_from_probs(info['val_probs'], val_future_rets, top_pct=CONFIG['long_only_top_pct'])
        cum_ret = float((1 + np.array(info['val_probs']) * val_future_rets).prod() - 1) if len(val_future_rets) > 0 else 0.0

        # Normalize Sharpe to [0,1] range (clip at -3,3)
        sharpe_norm  = (np.clip(sharpe,  -3, 3) + 3) / 6
        # Normalize IC to [0,1] range
        ic_norm      = (np.clip(ic, -1, 1) + 1) / 2

        score = (w.get('val_auc', 0.25)              * va
               + w.get('val_pr_auc', 0.15)           * pr
               + w.get('val_sharpe', 0.30)           * sharpe_norm
               + w.get('val_ic', 0.15)               * ic_norm
               + w.get('val_cumulative_return', 0.15)* max(0, min(1, 0.5 + cum_ret)))
        return float(score), float(sharpe), float(ic)

    val_future_rets = val_rows['future_return'].fillna(0).values

    # Align DL model val probs to val_rows future_return if sizes differ
    for name, info in model_registry.items():
        if info['type'] == 'dl':
            vp = info['val_probs']
            vr = val_future_rets[:len(vp)]
            s, sh, ic = composite_selection_score(
                {'val_probs': vp, 'val_auc': info['val_auc'],
                 'val_pr_auc': info.get('val_pr_auc', 0),
                 'val_ic': info.get('val_ic', 0)}, y_val, vr)
            info['selection_score'] = s
            info['val_sharpe_simulated'] = sh
        else:
            s, sh, ic = composite_selection_score(info, y_val, val_future_rets)
            info['selection_score'] = s
            info['val_sharpe_simulated'] = sh

    best_name = max(model_registry, key=lambda n: model_registry[n]['selection_score'])
    best_info = model_registry[best_name]

    print('\nModel Leaderboard (by composite score):')
    print(f'  {"Model":<22s}  {"Val AUC":>8s}  {"Val IC":>8s}  {"Sim Sharpe":>11s}  {"Score":>7s}')
    print('  ' + '-'*62)
    for n, info in sorted(model_registry.items(), key=lambda x: -x[1].get('selection_score', 0)):
        marker = '  <-- BEST' if n == best_name else ''
        print(f'  {n:<22s}  {info["val_auc"]:>8.4f}  '
              f'{info.get("val_ic", 0):>8.4f}  '
              f'{info.get("val_sharpe_simulated", 0):>11.4f}  '
              f'{info.get("selection_score", 0):>7.4f}{marker}')

    # ---- Optimal Threshold -------------------------------------------
    best_val_probs  = best_info['val_probs']
    best_test_probs = best_info['test_probs']

    # Align to val/test sizes
    if len(best_val_probs) > len(y_val):
        best_val_probs = best_val_probs[:len(y_val)]
    if len(best_test_probs) > len(y_test):
        best_test_probs = best_test_probs[:len(y_test)]

    thresholds = np.linspace(0.3, 0.7, 81)
    metric = CONFIG['threshold_metric']
    best_threshold = 0.5
    best_thresh_score = -np.inf
    for t in thresholds:
        preds = (best_val_probs >= t).astype(int)
        if preds.sum() == 0 or (1 - preds).sum() == 0:
            continue
        if metric == 'mcc':
            s = matthews_corrcoef(y_val[:len(preds)], preds)
        elif metric == 'f1':
            s = f1_score(y_val[:len(preds)], preds, zero_division=0)
        else:
            s = balanced_accuracy_score(y_val[:len(preds)], preds)
        if s > best_thresh_score:
            best_thresh_score = s
            best_threshold = float(t)

    test_preds    = (best_test_probs >= best_threshold).astype(int)
    best_model    = best_info['model']

    print(f'\nBest model:  {best_name}')
    print(f'Threshold:   {best_threshold:.4f} ({metric})')
    print(f'Test preds:  {test_preds.sum()} positive / {len(test_preds)} total')


# =======================================================================
# Cell 9B - REGRESSION MODELS (Next-Day Return & Price Prediction)
# =======================================================================

with cell_timer('Regression Models'):
    from sklearn.linear_model import ElasticNet, HuberRegressor, Ridge
    from sklearn.neural_network import MLPRegressor

    y_train_reg = train_rows['future_return'].astype(np.float32)
    y_val_reg   = val_rows['future_return'].astype(np.float32)
    y_test_reg  = test_rows['future_return'].astype(np.float32)

    reg_registry = {}
    reg_search   = []

    def dir_acc(y_true, y_pred):
        return float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    def safe_fit_reg(model, X, y, sw=None):
        try:
            model.fit(X, y, sample_weight=sw) if sw is not None else model.fit(X, y)
        except TypeError:
            model.fit(X, y)
        return model

    def register_reg(name, model, meta):
        vp = model.predict(X_val_model)
        tp = model.predict(X_test_model)
        reg_registry[name] = {
            'model': model,
            'val': vp, 'test': tp,
            'val_rmse': float(np.sqrt(mean_squared_error(y_val_reg, vp))),
            'val_da':   dir_acc(y_val_reg.values, vp),
            'meta': meta,
        }

    # Ridge
    best_rmse, best_model, best_meta = np.inf, None, None
    for alpha in [0.1, 1.0, 5.0, 10.0, 50.0]:
        m    = Ridge(alpha=alpha, random_state=CONFIG['random_seed'])
        safe_fit_reg(m, X_train_model, y_train_reg, sw=train_weights)
        pred = m.predict(X_val_model)
        rmse = float(np.sqrt(mean_squared_error(y_val_reg, pred)))
        reg_search.append({'model':'Ridge', 'alpha':alpha, 'val_rmse':rmse})
        if rmse < best_rmse:
            best_rmse, best_model, best_meta = rmse, m, {'alpha':alpha}
    register_reg('Ridge', best_model, best_meta)

    # ElasticNet
    best_rmse, best_model, best_meta = np.inf, None, None
    for alpha in [1e-4, 5e-4, 1e-3, 5e-3]:
        for l1 in [0.1, 0.5, 0.9]:
            m    = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=5000, random_state=CONFIG['random_seed'])
            safe_fit_reg(m, X_train_model, y_train_reg, sw=train_weights)
            pred = m.predict(X_val_model)
            rmse = float(np.sqrt(mean_squared_error(y_val_reg, pred)))
            reg_search.append({'model':'ElasticNet', 'alpha':alpha, 'l1_ratio':l1, 'val_rmse':rmse})
            if rmse < best_rmse:
                best_rmse, best_model, best_meta = rmse, m, {'alpha':alpha, 'l1_ratio':l1}
    register_reg('ElasticNet', best_model, best_meta)

    # Huber (robust to outliers)
    best_rmse, best_model, best_meta = np.inf, None, None
    for alpha in [1e-5, 1e-4, 1e-3]:
        for eps in [1.2, 1.35, 1.5]:
            m    = HuberRegressor(alpha=alpha, epsilon=eps, max_iter=500)
            safe_fit_reg(m, X_train_model, y_train_reg, sw=train_weights)
            pred = m.predict(X_val_model)
            rmse = float(np.sqrt(mean_squared_error(y_val_reg, pred)))
            reg_search.append({'model':'Huber', 'alpha':alpha, 'epsilon':eps, 'val_rmse':rmse})
            if rmse < best_rmse:
                best_rmse, best_model, best_meta = rmse, m, {'alpha':alpha, 'epsilon':eps}
    register_reg('HuberRegressor', best_model, best_meta)

    save_json_artifact(OUTPUT_DIRS['reports'] / 'regression_search.json', reg_search)

    best_reg_name  = min(reg_registry, key=lambda n: reg_registry[n]['val_rmse'])
    best_reg_info  = reg_registry[best_reg_name]
    test_reg_preds = best_reg_info['test']

    # Predict next-day close from best regressor
    predicted_next_close = (test_rows['Close'].values * (1 + np.clip(test_reg_preds, -0.5, 0.5)))

    print('\nRegressor Summary:')
    for n, info in sorted(reg_registry.items(), key=lambda x: x[1]['val_rmse']):
        marker = '  <-- BEST' if n == best_reg_name else ''
        print(f'  {n:<20s} Val RMSE: {info["val_rmse"]:.6f}  '
              f'Dir-Acc: {info["val_da"]:.4f}{marker}')


# =======================================================================
# Cell 10 - FULL MODEL EVALUATION + SIGNAL DECAY ANALYSIS
# =======================================================================

with cell_timer('Full Model Evaluation & Signal Decay'):

    # ---- Classification Report ----------------------------------------
    def full_cls_report(y_true, probs, preds, title):
        metrics = {
            'accuracy':          accuracy_score(y_true, preds),
            'balanced_accuracy': balanced_accuracy_score(y_true, preds),
            'precision':         precision_score(y_true, preds, zero_division=0),
            'recall':            recall_score(y_true, preds, zero_division=0),
            'f1':                f1_score(y_true, preds, zero_division=0),
            'roc_auc':           roc_auc_score(y_true, probs),
            'pr_auc':            average_precision_score(y_true, probs),
            'mcc':               matthews_corrcoef(y_true, preds),
            'cohen_kappa':       cohen_kappa_score(y_true, preds),
            'log_loss':          log_loss(y_true, probs),
            'brier_score':       brier_score_loss(y_true, probs),
        }
        print(f'\n{"="*60}\n{title}\n{"="*60}')
        for k, v in metrics.items():
            print(f'  {k:<25s}: {v:.4f}')
        print(f'\n{classification_report(y_true, preds, digits=4)}')
        return metrics

    cls_metrics = full_cls_report(
        y_test[:len(test_preds)], best_test_probs, test_preds,
        f'TEST CLASSIFICATION -- {best_name}'
    )

    # ---- Regression Report -------------------------------------------
    def full_reg_report(actual_close, pred_close, actual_ret, pred_ret, title):
        metrics = {
            'mae':      mean_absolute_error(actual_close, pred_close),
            'rmse':     np.sqrt(mean_squared_error(actual_close, pred_close)),
            'mape':     float(np.mean(np.abs((actual_close - pred_close) / (actual_close + 1e-9)))),
            'r2':       r2_score(actual_close, pred_close),
            'dir_acc':  float(np.mean(np.sign(actual_ret) == np.sign(pred_ret))),
        }
        print(f'\n{"="*60}\n{title}\n{"="*60}')
        for k, v in metrics.items():
            print(f'  {k:<25s}: {v:.4f}')
        return metrics

    reg_metrics = full_reg_report(
        test_rows['future_close'].values, predicted_next_close,
        y_test_reg.values, test_reg_preds,
        f'TEST REGRESSION -- {best_reg_name}'
    )

    # ---- Generalization Gap Analysis ---------------------------------
    print('\n' + '='*60)
    print('GENERALIZATION GAP ANALYSIS')
    print('='*60)
    gap_rows = []
    for name, info in model_registry.items():
        tp = info.get('train_probs')
        if tp is None or len(tp) != len(y_train):
            continue
        tr_auc  = roc_auc_score(y_train, tp)
        te_auc  = roc_auc_score(y_test[:len(info['test_probs'])], info['test_probs'][:len(y_test)])
        gap     = tr_auc - te_auc
        flag    = '  OVERFIT' if gap > 0.05 else '  OK'
        print(f'  {name:<22s} Train AUC: {tr_auc:.4f}  Test AUC: {te_auc:.4f}  Gap: {gap:+.4f}{flag}')
        gap_rows.append({'model':name, 'train_auc':tr_auc, 'test_auc':te_auc, 'gap':gap})
        info['generalization_gap'] = gap

    pd.DataFrame(gap_rows).to_csv(OUTPUT_DIRS['reports'] / 'generalization_gap.csv', index=False)

    # ---- Information Coefficient Analysis ----------------------------
    print('\n' + '='*60)
    print('INFORMATION COEFFICIENT ANALYSIS')
    print('='*60)

    ic_rows = []
    test_future_rets = test_rows['future_return'].fillna(0).values

    for name, info in model_registry.items():
        tp = info['test_probs'][:len(test_future_rets)]
        ic = information_coefficient(tp, test_future_rets[:len(tp)])
        print(f'  {name:<22s} Test IC: {ic:.4f}')
        info['test_ic'] = ic
        ic_rows.append({'model': name, 'test_ic': ic})

    # ---- Signal Decay (IC at multiple horizons) ----------------------
    print('\n' + '='*60)
    print('SIGNAL DECAY ANALYSIS (IC at Multiple Horizons)')
    print('='*60)

    horizons  = CONFIG['ic_horizons']
    decay_results = {}

    for name, info in model_registry.items():
        test_probs_decay = info['test_probs']
        horizon_ics = []
        for h in horizons:
            horizon_col = f'future_return_h{h}'
            if horizon_col not in test_rows.columns:
                fut_arr = np.full(len(test_rows), np.nan)
                for tkr, grp in test_rows.groupby('ticker', sort=False):
                    idx_arr = grp.index.to_numpy()
                    c_arr   = grp['Close'].to_numpy()
                    for local_i in range(len(c_arr) - h):
                        if c_arr[local_i] > 0:
                            fut_arr[idx_arr[local_i]] = c_arr[local_i + h] / c_arr[local_i] - 1
                rets = fut_arr
            else:
                rets = test_rows[horizon_col].values

            rets = rets[:len(test_probs_decay)]
            ic_h = information_coefficient(test_probs_decay[:len(rets)], rets)
            horizon_ics.append(ic_h)

        decay_results[name] = horizon_ics
        ics_str = '  '.join([f'H{h}:{ic:.3f}' for h, ic in zip(horizons, horizon_ics)])
        print(f'  {name:<22s} {ics_str}')

    # Signal Decay Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (name, ics) in enumerate(decay_results.items()):
        lw    = 2.5 if name == best_name else 1.5
        alpha = 1.0 if name == best_name else 0.7
        ax.plot(horizons, ics, 'o-', color=PAPER_COLORS[i % len(PAPER_COLORS)],
                lw=lw, alpha=alpha, label=name, markersize=7)
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.fill_between(horizons, [0]*len(horizons),
                    [max([decay_results[n][i] for n in decay_results]) for i in range(len(horizons))],
                    alpha=0.05, color='green')
    ax.set_xlabel('Prediction Horizon (Days)', fontsize=12)
    ax.set_ylabel('Information Coefficient (Spearman IC)', fontsize=12)
    ax.set_title('Signal Decay Analysis -- IC vs Prediction Horizon', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xticks(horizons)
    ax.set_xticklabels([f'H={h}d' for h in horizons])
    save_plot('signal_decay_analysis.png')
    print('\nSignal decay plot saved.')

    save_json_artifact(OUTPUT_DIRS['reports'] / 'ic_analysis.json', {
        'ic_by_model': ic_rows,
        'signal_decay': {n: list(map(float, ics)) for n, ics in decay_results.items()},
        'horizons': horizons,
    })


# =======================================================================
# Cell 11 - BACKTESTING  (Kelly Sizing + Regime-Conditional Performance)
# =======================================================================

with cell_timer('Backtesting'):

    # ---- Strategy Simulation Helpers ---------------------------------
    def compute_trading_metrics(returns, equity):
        ret = pd.Series(returns).fillna(0.0)
        eq  = pd.Series(equity).fillna(CONFIG['starting_capital'])
        if len(ret) < 2:
            return {}
        sharpe  = (ret.mean() / (ret.std() + 1e-9)) * np.sqrt(252)
        down    = ret[ret < 0].std()
        sortino = (ret.mean() / (down + 1e-9)) * np.sqrt(252)
        dd      = (eq - eq.cummax()) / (eq.cummax() + 1e-9)
        max_dd  = float(dd.min())
        ann_ret = ret.mean() * 252
        calmar  = ann_ret / (abs(max_dd) + 1e-9)
        gp      = ret[ret > 0].sum()
        gl      = -ret[ret < 0].sum()
        pf      = gp / (gl + 1e-9)
        years   = max(len(ret) / 252, 1/252)
        cagr    = (eq.iloc[-1] / (eq.iloc[0] + 1e-9)) ** (1/years) - 1
        cv      = cvar(ret.values)
        om      = omega_ratio(ret.values)

        return {
            'sharpe_ratio':       float(sharpe),
            'sortino_ratio':      float(sortino),
            'calmar_ratio':       float(calmar),
            'cagr':               float(cagr),
            'max_drawdown':       max_dd,
            'cvar_5pct':          float(cv),
            'omega_ratio':        min(float(om), 10.0), 
            'profit_factor':      float(pf),
            'win_rate':           float((ret > 0).mean()),
            'annual_volatility':  float(ret.std() * np.sqrt(252)),
            'avg_daily_return':   float(ret.mean()),
            'cumulative_return':  float(eq.iloc[-1] / (eq.iloc[0] + 1e-9) - 1),
        }

    def simulate_ranked_strategy(rows, probs, strategy_name, long_top_pct=0.30, short_bottom_pct=0.0, smooth_span=None, target_vol=None):
        bt  = rows.copy()
        bt['prob'] = probs[:len(bt)]
        daily_rows = []

        for date, grp in bt.groupby('Date', sort=True):
            n         = len(grp)
            n_long    = max(1, int(n * long_top_pct))
            n_short   = max(1, int(n * short_bottom_pct)) if short_bottom_pct > 0 else 0
            long_idx  = grp['prob'].nlargest(n_long).index
            short_idx = grp['prob'].nsmallest(n_short).index if n_short > 0 else []

            for idx in long_idx:
                ret  = float(rows.loc[idx, 'future_return'] if idx in rows.index else 0)
                mret = float(rows.loc[idx, 'market_future_return'] if idx in rows.index else 0)
                pos  = 1.0 / n_long
                pos  = min(pos, CONFIG['max_position_concentration'])
                daily_rows.append({
                    'Date': date, 'ticker': grp.loc[idx, 'ticker'] if idx in grp.index else '',
                    'pos': pos, 'side': 1,
                    'asset_return':        pos * ret,
                    'turnover':            abs(pos - 0.0),
                    'market_future_return': mret,
                })
            for idx in short_idx:
                ret  = float(rows.loc[idx, 'future_return'] if idx in rows.index else 0)
                mret = float(rows.loc[idx, 'market_future_return'] if idx in rows.index else 0)
                pos  = 1.0 / max(n_short, 1)
                pos  = min(pos, CONFIG['max_position_concentration'])
                daily_rows.append({
                    'Date': date, 'ticker': grp.loc[idx, 'ticker'] if idx in grp.index else '',
                    'pos': pos, 'side': -1,
                    'asset_return':        -pos * ret,
                    'turnover':            abs(pos - 0.0),
                    'market_future_return': mret,
                })

        if not daily_rows:
            empty = pd.DataFrame()
            return empty, empty, {}

        bt_detail = pd.DataFrame(daily_rows)

        # Daily aggregation
        daily = bt_detail.groupby('Date', as_index=False).agg(
            port_ret=('asset_return',        'sum'),
            turnover=('turnover',            'sum'),
            mkt_ret =('market_future_return', 'first'),
        )

        if target_vol is not None and smooth_span is not None:
            daily['smooth_ret'] = daily['port_ret'].ewm(span=smooth_span).mean()
            daily['roll_vol']   = daily['port_ret'].rolling(20).std().shift(1)
            daily['vol_scale']  = (target_vol / (daily['roll_vol'] + 1e-9)).clip(0, 2.5).fillna(1.0)
        else:
            daily['vol_scale'] = 1.0

        daily['turnover']  = daily['turnover'].clip(upper=CONFIG['max_daily_turnover'])
        daily['cost']      = daily['turnover'] * daily['vol_scale'] * CONFIG['transaction_cost']
        daily['slippage']  = daily['turnover'] * CONFIG['slippage_factor']
        daily['strat_ret'] = daily['port_ret'] * daily['vol_scale'] - daily['cost'] - daily['slippage']
        daily['equity']    = CONFIG['starting_capital'] * (1 + daily['strat_ret']).cumprod()
        daily['mkt_equity']= CONFIG['starting_capital'] * (1 + daily['mkt_ret']).cumprod()
        daily['cum_strat'] = daily['equity'] / CONFIG['starting_capital']
        daily['cum_mkt']   = daily['mkt_equity'] / CONFIG['starting_capital']
        daily['drawdown']  = (daily['equity'] - daily['equity'].cummax()) / (daily['equity'].cummax() + 1e-9)

        metrics = compute_trading_metrics(daily['strat_ret'], daily['equity'])
        return bt_detail, daily, metrics

    def simulate_signal_strategy(rows, probs, strategy_name, buy_threshold=0.55, sell_threshold=0.45):
        positions = {}
        daily_rows = []
        bt_sig = rows.copy()
        bt_sig['prob'] = probs[:len(rows)]
        for date, grp in bt_sig.groupby('Date', sort=True):
            n_buy    = int(grp['prob'].ge(buy_threshold).sum())
            pos_size = (min(1.0 / max(n_buy, 1), CONFIG['max_position_concentration']) if n_buy > 0 else 0.0)
            for ticker, sub in grp.groupby('ticker', sort=False):
                row      = sub.iloc[0]
                prob     = float(row['prob'])
                ret      = float(row.get('future_return', 0) or 0)
                mret     = float(row.get('market_future_return', 0) or 0)
                old_pos  = positions.get(ticker, 0.0)
                if prob >= buy_threshold:
                    new_pos = pos_size
                elif prob <= sell_threshold and old_pos > 0:
                    new_pos = 0.0
                else:
                    new_pos = old_pos
                new_pos  = min(new_pos, CONFIG['max_position_concentration'])
                turnover = abs(new_pos - old_pos)
                positions[ticker] = new_pos
                daily_rows.append({
                    'Date': date, 'ticker': ticker,
                    'asset_return': new_pos * ret,
                    'turnover': turnover,
                    'market_future_return': mret,
                })
        if not daily_rows:
            empty = pd.DataFrame()
            return empty, empty, {}
        
        detail = pd.DataFrame(daily_rows)
        daily = detail.groupby('Date', as_index=False).agg(
            port_ret=('asset_return', 'sum'),
            turnover=('turnover', 'sum'),
            mkt_ret =('market_future_return', 'first')
        )
        daily['turnover']  = daily['turnover'].clip(upper=CONFIG['max_daily_turnover'])
        daily['cost']      = daily['turnover'] * CONFIG['transaction_cost']
        daily['slippage']  = daily['turnover'] * CONFIG['slippage_factor']
        daily['strat_ret'] = daily['port_ret'] - daily['cost'] - daily['slippage']
        daily['equity']    = CONFIG['starting_capital'] * (1 + daily['strat_ret']).cumprod()
        daily['mkt_equity']= CONFIG['starting_capital'] * (1 + daily['mkt_ret']).cumprod()
        daily['cum_strat'] = daily['equity'] / CONFIG['starting_capital']
        daily['cum_mkt']   = daily['mkt_equity'] / CONFIG['starting_capital']
        daily['drawdown']  = (daily['equity'] - daily['equity'].cummax()) / (daily['equity'].cummax() + 1e-9)
        metrics = compute_trading_metrics(daily['strat_ret'], daily['equity'])
        return detail, daily, metrics

    def apply_drawdown_reduction(daily_df):
        df  = daily_df.copy()
        if df.empty:
            return df
        pk  = df['equity'].cummax()
        dd  = (df['equity'] - pk) / (pk + 1e-9)
        mask = dd.shift(1).fillna(0) < CONFIG['drawdown_reduction_threshold']
        df.loc[mask, 'strat_ret'] *= CONFIG['drawdown_reduction_factor']
        df['equity']    = CONFIG['starting_capital'] * (1 + df['strat_ret']).cumprod()
        df['cum_strat'] = df['equity'] / CONFIG['starting_capital']
        df['drawdown']  = (df['equity'] - df['equity'].cummax()) / (df['equity'].cummax() + 1e-9)
        return df

    def print_metrics(m, title):
        print(f'\n{"="*60}\n{title}\n{"="*60}')
        fmt_pct = {'cumulative_return', 'max_drawdown', 'cvar_5pct', 'win_rate', 'annual_volatility', 'cagr'}
        for k, v in m.items():
            if k in fmt_pct:
                print(f'  {k:<25s}: {v:>10.2%}')
            else:
                print(f'  {k:<25s}: {v:>10.4f}')

    # ---- Run Strategies -----------------------------------------------
    long_bt,   long_df,   long_metrics   = simulate_ranked_strategy(
        test_rows, best_test_probs, 'Long-Only', long_top_pct=CONFIG['long_only_top_pct']
    )

    ls_bt,     ls_df,     ls_metrics     = simulate_ranked_strategy(
        test_rows, best_test_probs, 'Long-Short',
        long_top_pct=CONFIG['long_short_top_pct'], short_bottom_pct=CONFIG['long_short_bottom_pct'],
        smooth_span=CONFIG['long_short_smoothing_span'], target_vol=CONFIG['long_short_target_vol']
    )

    sig_bt, signal_df, signal_metrics = simulate_signal_strategy(
        test_rows, best_test_probs, 'Signal',
        buy_threshold=max(CONFIG['buy_signal_threshold'], best_threshold),
        sell_threshold=min(CONFIG['sell_signal_threshold'], 1 - best_threshold)
    )

    long_df   = apply_drawdown_reduction(long_df)
    ls_df     = apply_drawdown_reduction(ls_df)
    signal_df = apply_drawdown_reduction(signal_df)

    long_metrics   = compute_trading_metrics(long_df['strat_ret'],   long_df['equity']) if not long_df.empty else {}
    ls_metrics     = compute_trading_metrics(ls_df['strat_ret'],     ls_df['equity']) if not ls_df.empty else {}
    signal_metrics = compute_trading_metrics(signal_df['strat_ret'], signal_df['equity']) if not signal_df.empty else {}

    print_metrics(long_metrics,   'LONG-ONLY STRATEGY')
    print_metrics(ls_metrics,     'LONG-SHORT STRATEGY (Vol-Targeted)')
    print_metrics(signal_metrics, 'BUY/SELL SIGNAL STRATEGY')

    # ---- Regime-Conditional Performance ----------------------------------
    print('\n' + '='*60)
    print('REGIME-CONDITIONAL BACKTEST PERFORMANCE')
    print('='*60)

    regime_perf = {}
    if 'regime_name' in test_rows.columns:
        for regime_name, regime_grp in test_rows.groupby('regime_name', sort=False):
            r_dates = set(regime_grp['Date'].unique())
            r_long  = long_df[long_df['Date'].isin(r_dates)].copy() if not long_df.empty else pd.DataFrame()
            if len(r_long) < 5:
                continue
            r_metrics = compute_trading_metrics(r_long['strat_ret'], r_long['equity'])
            regime_perf[str(regime_name)] = r_metrics
            print(f'  {str(regime_name):<25s} '
                  f'Sharpe: {r_metrics.get("sharpe_ratio", 0):6.2f}  '
                  f'CumRet: {r_metrics.get("cumulative_return", 0):7.2%}  '
                  f'MaxDD:  {r_metrics.get("max_drawdown", 0):7.2%}')

    # ---- Kelly Sizing Estimate ----------------------------------------
    if not long_df.empty:
        ret_series = long_df['strat_ret'].values
        wins   = ret_series[ret_series > 0]
        losses = np.abs(ret_series[ret_series < 0])
        if len(wins) > 0 and len(losses) > 0:
            wr    = len(wins) / (len(wins) + len(losses))
            avg_w = wins.mean()
            avg_l = losses.mean()
            k     = compute_kelly(wr, avg_w, avg_l) * CONFIG['kelly_fraction']
            print(f'\n  Kelly fraction (1/{int(1/CONFIG["kelly_fraction"])} Kelly): {k:.4f}')
            print(f'  Win rate: {wr:.2%}  Avg-Win: {avg_w:.4%}  Avg-Loss: {avg_l:.4%}')

    # Save trade books
    for bt, fname in [(long_bt, 'long_only_trades.csv'), (ls_bt, 'long_short_trades.csv'), (sig_bt, 'signal_trades.csv')]:
        if not bt.empty:
            bt.to_csv(OUTPUT_DIRS['backtests'] / fname, index=False)
    for df, fname in [(long_df, 'long_only_equity.csv'), (ls_df, 'long_short_equity.csv'), (signal_df, 'signal_equity.csv')]:
        if not df.empty:
            df.to_csv(OUTPUT_DIRS['backtests'] / fname, index=False)

    backtest_summary = {
        'best_model': best_name,
        'long_only':  long_metrics,
        'long_short': ls_metrics,
        'signal':     signal_metrics,
        'regime_conditional': regime_perf,
    }
    save_json_artifact(OUTPUT_DIRS['backtests'] / 'backtest_summary.json', backtest_summary)
    print('\nBacktest complete. Trade books & equity curves saved.')


# =======================================================================
# Cell 12 - EXPLAINABILITY & FEATURE IMPORTANCE (SHAP + Permutation)
# =======================================================================

with cell_timer('Explainability & Feature Importance'):

    def compute_importance(model_name, info, X_ref, y_ref):
        explicit = info.get('feature_importance')
        if explicit is not None and len(explicit) == len(feature_cols):
            return np.asarray(explicit, dtype=float)
        model = info['model']
        if hasattr(model, 'coef_'):
            return np.abs(np.asarray(model.coef_)).reshape(-1)
        if hasattr(model, 'coefs_') and model.coefs_:
            return np.mean(np.abs(np.asarray(model.coefs_[0])), axis=1)
        n   = min(len(X_ref), 1000)
        smp = X_ref.sample(n, random_state=CONFIG['random_seed'])
        tgt = y_ref.loc[smp.index]
        pr  = permutation_importance(model, smp, tgt, scoring='roc_auc', n_repeats=5, random_state=CONFIG['random_seed'], n_jobs=1)
        return np.asarray(pr.importances_mean, dtype=float)

    fi_store = {}
    for name, info in model_registry.items():
        if info['type'] == 'dl':
            continue
        try:
            fi_store[name] = compute_importance(name, info, X_val_model, y_val)
        except Exception as e:
            print(f'  Importance failed for {name}: {e}')

    explainable = {n: v for n, v in fi_store.items() if v is not None and len(v) == len(feature_cols) and np.any(np.abs(v) > 0)}
    feature_importance_store = explainable

    if explainable:
        # Choose best classical model for primary importance plot
        fi_model_name = (best_name if best_name in explainable else max(explainable, key=lambda n: model_registry[n].get('selection_score', 0)))

        imp_vec = np.nan_to_num(explainable[fi_model_name])
        imp_df  = pd.DataFrame({'feature': feature_cols, 'importance': imp_vec}).sort_values('importance', ascending=False)
        imp_df.to_csv(OUTPUT_DIRS['reports'] / 'feature_importance.csv', index=False)

        # Top-20 bar
        top20 = imp_df.head(20)
        fig, ax = plt.subplots(figsize=(10, 7))
        colors  = [PAPER_COLORS[0] if v > imp_vec.mean() else PAPER_COLORS[7] for v in top20['importance']]
        ax.barh(top20['feature'][::-1], top20['importance'][::-1], color=colors[::-1], edgecolor='white')
        ax.set_xlabel('Feature Importance')
        ax.set_title(f'Top-20 Feature Importances ({fi_model_name})', fontweight='bold')
        ax.axvline(imp_vec.mean(), color='red', ls='--', alpha=0.5, label=f'Mean={imp_vec.mean():.4f}')
        ax.legend(fontsize=9)
        save_plot('feature_importance.png')

        # SHAP analysis
        if shap is not None:
            try:
                fi_model    = model_registry[fi_model_name]['model']
                bg_n        = min(len(X_train_model), CONFIG['shap_background_size'])
                smp_n       = min(len(X_val_model),   CONFIG['shap_sample_size'])
                background  = X_train_model.sample(bg_n,  random_state=CONFIG['random_seed'])
                explain_df  = X_val_model.sample(smp_n, random_state=CONFIG['random_seed'])

                if hasattr(fi_model, 'coef_'):
                    explainer   = shap.LinearExplainer(fi_model, background)
                    shap_values = explainer.shap_values(explain_df)
                    if isinstance(shap_values, list):
                        shap_values = shap_values[-1]
                else:
                    explainer   = shap.Explainer(
                        lambda arr: fi_model.predict_proba(pd.DataFrame(arr, columns=feature_cols))[:, 1],
                        background.to_numpy()
                    )
                    shap_result = explainer(explain_df.to_numpy())
                    shap_values = np.asarray(shap_result.values)

                shap_values = np.asarray(shap_values, dtype=float)
                if shap_values.ndim == 2 and shap_values.shape[1] == len(feature_cols):
                    shap.summary_plot(shap_values, explain_df, feature_names=feature_cols, plot_type='bar', show=False, max_display=20)
                    save_plot('shap_bar.png')
                    shap.summary_plot(shap_values, explain_df, feature_names=feature_cols, show=False, max_display=20)
                    save_plot('shap_beeswarm.png')
                    print('  SHAP plots saved.')

                shap_df = pd.DataFrame(np.abs(shap_values), columns=feature_cols).mean()
                shap_df.sort_values(ascending=False).head(20).to_csv(OUTPUT_DIRS['reports'] / 'shap_importance.csv')
            except Exception as e:
                print(f'  SHAP failed: {e}')
        else:
            print('  SHAP not installed. Skipping SHAP analysis.')
    else:
        print('No classical model produced a usable feature-importance vector.')


# =======================================================================
# Cell 13 - CORE VISUALIZATIONS
# =======================================================================

with cell_timer('Core Visualizations'):

    # ---- 1. Confusion Matrix -----------------------------------------
    cm = confusion_matrix(y_test[:len(test_preds)], test_preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    pct_cm  = cm / cm.sum() * 100
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, linewidths=0.5)
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.7, f'({pct_cm[i,j]:.1f}%)', ha='center', va='center', fontsize=9, color='gray')
    ax.set_title(f'Confusion Matrix -- {best_name}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticklabels(['DOWN', 'UP'])
    ax.set_yticklabels(['DOWN', 'UP'])
    save_plot('confusion_matrix.png')

    # ---- 2. ROC + PR Curves (all models) ----------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, (name, info) in enumerate(sorted(model_registry.items(), key=lambda x: -x[1].get('val_auc', 0))):
        tp  = info['test_probs'][:len(y_test)]
        fpr, tpr, _ = roc_curve(y_test[:len(tp)], tp)
        auc = roc_auc_score(y_test[:len(tp)], tp)
        lw  = 2.5 if name == best_name else 1.0
        axes[0].plot(fpr, tpr, label=f'{name} ({auc:.4f})', lw=lw, color=PAPER_COLORS[i % len(PAPER_COLORS)])
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
    axes[0].set_title('ROC Curves -- All Models', fontweight='bold')
    axes[0].set_xlabel('FPR')
    axes[0].set_ylabel('TPR')
    axes[0].legend(fontsize=8)

    prec, rec, _ = precision_recall_curve(y_test[:len(best_test_probs)], best_test_probs)
    pr_auc        = average_precision_score(y_test[:len(best_test_probs)], best_test_probs)
    axes[1].plot(rec, prec, color='teal', lw=2, label=f'{best_name} (PR-AUC={pr_auc:.4f})')
    axes[1].axhline(y_test.mean(), color='gray', ls='--', alpha=0.5, label='Baseline')
    axes[1].set_title(f'Precision-Recall -- {best_name}', fontweight='bold')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].legend(fontsize=9)
    save_plot('roc_pr_curves.png')

    # ---- 3. Next-Day Price Prediction --------------------------------
    reg_plot = test_rows[['Date', 'ticker', 'future_close']].copy()
    reg_plot['pred_close'] = predicted_next_close
    focus = reg_plot['ticker'].value_counts().index[0]
    fdf   = reg_plot[reg_plot['ticker']==focus].tail(120)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(fdf['Date'], fdf['future_close'], lw=2, label=f'{focus} Actual', color=PAPER_COLORS[0])
    ax.plot(fdf['Date'], fdf['pred_close'],   lw=2, linestyle='--', label=f'{focus} Predicted ({best_reg_name})', color=PAPER_COLORS[1], alpha=0.85)
    ax.fill_between(fdf['Date'], fdf['future_close'],  fdf['pred_close'], alpha=0.08, color='gray')
    ax.set_title(f'Next-Day Close Price Prediction -- {focus}', fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price ($)')
    ax.legend()
    ax.tick_params(axis='x', rotation=30)
    save_plot('next_close_prediction.png')

    # ---- 4. Strategy Dashboard (2x2) --------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    # Cumulative returns
    if not long_df.empty:
        axes[0,0].plot(long_df['Date'], long_df['cum_strat'], lw=2, color='teal', label='Long-Only')
        axes[0,0].plot(long_df['Date'], long_df['cum_mkt'],   lw=1.5, color='gray', alpha=0.7, label='SPY')
        axes[0,0].axhline(1.0, color='black', ls=':', alpha=0.3)
        axes[0,0].set_title('Long-Only vs Market', fontweight='bold')
        axes[0,0].legend()
        axes[0,0].set_ylabel('Cumulative Return')

    if not ls_df.empty and not signal_df.empty:
        axes[0,1].plot(ls_df['Date'],     ls_df['equity'],     lw=2, color='coral',     label='Long-Short')
        axes[0,1].plot(signal_df['Date'], signal_df['equity'], lw=1.8, color='goldenrod', label='Signal')
        axes[0,1].plot(long_df['Date'],   long_df['mkt_equity'],lw=1.2, color='gray', alpha=0.6, label='SPY')
        axes[0,1].set_title('Strategy Equity Curves', fontweight='bold')
        axes[0,1].legend()
        axes[0,1].set_ylabel('Portfolio Value ($)')

    if not long_df.empty:
        axes[1,0].fill_between(long_df['Date'], long_df['drawdown'], 0, color='crimson', alpha=0.35, label='Long-Only DD')
    if not ls_df.empty:
        axes[1,0].plot(ls_df['Date'], ls_df['drawdown'], color='darkorange', lw=1, alpha=0.7, label='L/S DD')
    axes[1,0].set_title('Drawdown Profiles', fontweight='bold')
    axes[1,0].legend()
    axes[1,0].set_ylabel('Drawdown')

    # Best stock price overlay
    focus_all = test_rows[test_rows['ticker']==focus].tail(200).copy()
    focus_positions = test_rows.index.get_indexer(focus_all.index)
    valid_pos  = focus_positions[focus_positions >= 0]
    valid_rows = focus_all.iloc[focus_positions >= 0].copy()
    focus_probs = best_test_probs[valid_pos] if len(valid_pos) else np.array([])
    
    axes[1,1].plot(focus_all['Date'], focus_all['Close'], color='steelblue', lw=1.5)
    if len(focus_probs):
        buy_mask = focus_probs >= best_threshold
        axes[1,1].scatter(valid_rows['Date'].values[buy_mask], valid_rows['Close'].values[buy_mask],
                          color='green', s=20, zorder=5, alpha=0.7, label='Buy Signal')
    axes[1,1].set_title(f'Price + Buy Signals -- {focus}', fontweight='bold')
    axes[1,1].legend(fontsize=9)
    axes[1,1].set_ylabel('Price ($)')

    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=25)
    save_plot('strategy_dashboard.png')

    # ---- 5. Model Comparison Bar --------------------------------
    comp_df = pd.DataFrame([
        {'Model': n,
         'Val AUC': info['val_auc'],
         'Test AUC': roc_auc_score(y_test[:len(info['test_probs'])], info['test_probs'][:len(y_test)]),
         'Val IC':   info.get('val_ic', 0),
         'Score':    info.get('selection_score', 0)}
        for n, info in model_registry.items()
    ]).sort_values('Score', ascending=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    x = np.arange(len(comp_df)); w = 0.35

    axes[0].bar(x-w/2, comp_df['Val AUC'],  w, label='Val AUC',  color=PAPER_COLORS[0], edgecolor='white')
    axes[0].bar(x+w/2, comp_df['Test AUC'], w, label='Test AUC', color=PAPER_COLORS[1], edgecolor='white')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(comp_df['Model'], rotation=20)
    axes[0].set_title('Val vs Test AUC', fontweight='bold')
    axes[0].axhline(0.5, color='gray', ls='--', alpha=0.4)
    axes[0].legend()

    axes[1].bar(x, comp_df['Val IC'], color=PAPER_COLORS[3], edgecolor='white')
    axes[1].axhline(0, color='gray', ls='--', alpha=0.5)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(comp_df['Model'], rotation=20)
    axes[1].set_title('Information Coefficient (IC)', fontweight='bold')
    axes[1].set_ylabel('Spearman IC')

    axes[2].bar(x, comp_df['Score'], color=PAPER_COLORS[4], edgecolor='white')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(comp_df['Model'], rotation=20)
    axes[2].set_title('Composite Selection Score', fontweight='bold')
    axes[2].set_ylabel('Score')

    save_plot('model_comparison.png')
    print('Core visualizations saved.')


# =======================================================================
# Cell 14 - RESEARCH PAPER VISUALIZATIONS (28 Publication-Quality Figs)
# =======================================================================

with cell_timer('Research Paper Visualizations (28 Figures)'):

    print('='*60)
    print('GENERATING 28 RESEARCH PAPER VISUALIZATIONS')
    print('='*60)

    # [1] Feature Correlation Heatmap
    print('\n[1/28] Feature Correlation Heatmap...')
    top_n_corr  = min(30, len(feature_cols))
    corr_mat    = train_rows[feature_cols[:top_n_corr]].corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    mask    = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
    sns.heatmap(corr_mat, mask=mask, cmap='RdBu_r', center=0, square=True,
                linewidths=0.5, annot=False, ax=ax, cbar_kws={'label':'Pearson Correlation', 'shrink':0.8})
    ax.set_title('Feature Correlation Matrix (Top-30)', fontsize=16, fontweight='bold', pad=20)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
    save_plot('research_01_correlation_heatmap.png')

    # [2] Multi-Model Feature Importance Comparison
    print('[2/28] Multi-Model Feature Importance...')
    if feature_importance_store:
        fi_comp = pd.DataFrame(
            {n: np.nan_to_num(v) / (np.nan_to_num(v).max() + 1e-9) for n, v in feature_importance_store.items()},
            index=feature_cols
        )
        fi_comp['avg'] = fi_comp.mean(axis=1)
        top20_fi = fi_comp.nlargest(20, 'avg').drop(columns='avg')
        fig, ax = plt.subplots(figsize=(12, 8))
        top20_fi.plot(kind='barh', ax=ax, width=0.8, color=PAPER_COLORS[:len(fi_comp.columns)-1])
        ax.set_xlabel('Normalized Feature Importance')
        ax.set_title('Feature Importance Across Classical Models', fontsize=14, fontweight='bold')
        ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
        ax.invert_yaxis()
        save_plot('research_02_feature_importance_comparison.png')

    # [3] Learning Curves
    print('[3/28] Learning Curves...')
    lc_model_name = next((n for n, i in model_registry.items() if i['type']=='classical'), None)
    if lc_model_name:
        try:
            lc_m = clone(model_registry[lc_model_name]['model'])
            tsz, trs, vls = learning_curve(
                lc_m, X_train_model, y_train, train_sizes=np.linspace(0.1, 1.0, 8),
                cv=TimeSeriesSplit(n_splits=3), scoring='roc_auc', n_jobs=1
            )
            fig, ax = plt.subplots(figsize=(10, 6))
            tm, ts = trs.mean(1), trs.std(1)
            vm, vs = vls.mean(1), vls.std(1)
            ax.fill_between(tsz, tm-ts, tm+ts, alpha=0.15, color=PAPER_COLORS[0])
            ax.fill_between(tsz, vm-vs, vm+vs, alpha=0.15, color=PAPER_COLORS[1])
            ax.plot(tsz, tm, 'o-', color=PAPER_COLORS[0], lw=2, label='Train')
            ax.plot(tsz, vm, 's-', color=PAPER_COLORS[1], lw=2, label='CV Val')
            gap = tm[-1] - vm[-1]
            ax.annotate(f'Gap={gap:.4f}', xy=(tsz[-1], (tm[-1]+vm[-1])/2),
                        fontsize=11, ha='right', color='red' if gap > 0.05 else 'green', fontweight='bold')
            ax.set_xlabel('Training Set Size')
            ax.set_ylabel('ROC-AUC')
            ax.set_title(f'Learning Curves -- {lc_model_name}', fontsize=14, fontweight='bold')
            ax.legend(loc='lower right')
            save_plot('research_03_learning_curves.png')
        except Exception as e:
            print(f'  Learning curves failed: {e}')

    # [4] Calibration Plot
    print('[4/28] Calibration Plot...')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), gridspec_kw={'height_ratios':[3, 1]})
    for i, (name, info) in enumerate(model_registry.items()):
        try:
            tp = info['test_probs'][:len(y_test)]
            pt, pp = calibration_curve(y_test[:len(tp)], tp, n_bins=15, strategy='uniform')
            ax1.plot(pp, pt, 's-', label=name, lw=1.5, markersize=5, color=PAPER_COLORS[i % len(PAPER_COLORS)])
        except Exception:
            pass
    ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax1.set_xlabel('Mean Predicted Prob')
    ax1.set_ylabel('Fraction of Positives')
    ax1.set_title('Calibration Plot (Reliability Diagram)', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.set_xlim([0, 1])
    ax1.set_ylim([0, 1])
    ax2.hist(best_test_probs, bins=50, color=PAPER_COLORS[0], alpha=0.7, edgecolor='white')
    ax2.set_xlabel('Predicted Probability')
    ax2.set_ylabel('Count')
    save_plot('research_04_calibration_plot.png')

    # [5] Threshold Optimization
    print('[5/28] Threshold Optimization...')
    thr_range = np.linspace(0.1, 0.9, 161)
    f1_t, mcc_t, acc_t, ba_t = [], [], [], []
    for t in thr_range:
        pt = (best_test_probs[:len(y_test)] >= t).astype(int)
        f1_t.append(f1_score(y_test, pt, zero_division=0))
        mcc_t.append(matthews_corrcoef(y_test, pt))
        acc_t.append(accuracy_score(y_test, pt))
        ba_t.append(balanced_accuracy_score(y_test, pt))
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(thr_range, f1_t,  lw=2, label='F1',           color=PAPER_COLORS[0])
    ax.plot(thr_range, mcc_t, lw=2, label='MCC',          color=PAPER_COLORS[1])
    ax.plot(thr_range, acc_t, lw=2, label='Accuracy',     color=PAPER_COLORS[2])
    ax.plot(thr_range, ba_t,  lw=2, label='Balanced Acc', color=PAPER_COLORS[3])
    ax.axvline(best_threshold, color='red', ls='--', alpha=0.7, label=f'Optimal={best_threshold:.3f}')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Threshold Optimization Curve', fontsize=14, fontweight='bold')
    ax.legend()
    ax.set_xlim([0.1, 0.9])
    save_plot('research_05_threshold_optimization.png')

    # [6] Precision-Recall-F1 Trade-off
    print('[6/28] Precision-Recall-F1 Trade-off...')
    prec_t, rec_t, f1_t2 = [], [], []
    for t in thr_range:
        pt = (best_test_probs[:len(y_test)] >= t).astype(int)
        prec_t.append(precision_score(y_test, pt, zero_division=0))
        rec_t.append(recall_score(y_test, pt, zero_division=0))
        f1_t2.append(f1_score(y_test, pt, zero_division=0))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(thr_range, prec_t, lw=2, label='Precision', color=PAPER_COLORS[0])
    ax.plot(thr_range, rec_t,  lw=2, label='Recall',    color=PAPER_COLORS[1])
    ax.plot(thr_range, f1_t2,  lw=2, label='F1',        color=PAPER_COLORS[3])
    ax.axvline(best_threshold, color='red', ls='--', alpha=0.7)
    ax.fill_between(thr_range, prec_t, rec_t, alpha=0.07, color='gray')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Precision-Recall-F1 Trade-off', fontsize=14, fontweight='bold')
    ax.legend()
    save_plot('research_06_pr_f1_tradeoff.png')

    # [7] Cumulative Returns -- All Models
    print('[7/28] Cumulative Returns (All Models)...')
    fig, ax = plt.subplots(figsize=(14, 7))
    for i, (name, info) in enumerate(model_registry.items()):
        try:
            if name == best_name and not long_df.empty:
                daily_i = long_df
            else:
                _, daily_i, _ = simulate_ranked_strategy(test_rows, info['test_probs'], strategy_name=name, long_top_pct=CONFIG['long_only_top_pct'])
            if daily_i.empty:
                continue
            lw = 2.5 if name==best_name else 1.2
            ls = '-' if name==best_name else '--'
            ax.plot(daily_i['Date'], daily_i['cum_strat'], ls, label=name, lw=lw, color=PAPER_COLORS[i%len(PAPER_COLORS)], alpha=0.9)
        except Exception:
            pass
    if not long_df.empty:
        ax.plot(long_df['Date'], long_df['cum_mkt'], 'k-', lw=1.5, alpha=0.5, label='SPY')
    ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('Date')
    ax.set_ylabel('Cumulative Return')
    ax.set_title('Cumulative Returns -- All Models vs Market', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    save_plot('research_07_cumulative_returns_all_models.png')

    # [8] Rolling Sharpe
    print('[8/28] Rolling Sharpe...')
    fig, ax = plt.subplots(figsize=(14, 6))
    if not long_df.empty:
        for window, color, ls in [(60, PAPER_COLORS[0], '-'), (120, PAPER_COLORS[1], '--')]:
            rm = long_df['strat_ret'].rolling(window).mean()
            rs = long_df['strat_ret'].rolling(window).std()
            ax.plot(long_df['Date'], (rm/(rs+1e-9))*np.sqrt(252), ls, color=color, lw=1.8, label=f'{window}d Rolling Sharpe')
        ax.axhline(0, color='gray', ls=':', alpha=0.5)
        ax.axhline(1, color='green', ls=':', alpha=0.3, label='Sharpe=1')
        ax.axhline(-1, color='red', ls=':', alpha=0.3, label='Sharpe=-1')
        ax.set_xlabel('Date')
        ax.set_ylabel('Sharpe Ratio')
        ax.set_title('Rolling Sharpe Ratio -- Long-Only', fontsize=14, fontweight='bold')
        ax.legend()
        ax.tick_params(axis='x', rotation=30)
        save_plot('research_08_rolling_sharpe.png')

    # [9] Monthly Returns Heatmap
    print('[9/28] Monthly Returns Heatmap...')
    try:
        mon = long_df.copy()
        mon['Date'] = pd.to_datetime(mon['Date'])
        mon['Year'] = mon['Date'].dt.year
        mon['Month']= mon['Date'].dt.month
        mon_ret = mon.groupby(['Year', 'Month'])['strat_ret'].sum().unstack()
        mon_ret.columns = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][:len(mon_ret.columns)]
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.heatmap(mon_ret*100, annot=True, fmt='.1f', cmap='RdYlGn', center=0, linewidths=0.5, ax=ax, cbar_kws={'label':'Return (%)'})
        ax.set_title('Monthly Strategy Returns (%)', fontsize=14, fontweight='bold')
        ax.set_xlabel('Month')
        ax.set_ylabel('Year')
        save_plot('research_09_monthly_heatmap.png')
    except Exception as e:
        print(f'  Monthly heatmap failed: {e}')

    # [10] Feature Drift (PSI)
    print('[10/28] Feature Drift PSI...')
    def psi(ref, cur, bins=10):
        ref = pd.Series(ref).replace([np.inf, -np.inf], np.nan).dropna()
        cur = pd.Series(cur).replace([np.inf, -np.inf], np.nan).dropna()
        if ref.empty or cur.empty:
            return 0.0
        edges = np.unique(np.nanpercentile(ref, np.linspace(0, 100, bins+1)))
        if len(edges) <= 2:
            return 0.0
        rh, _ = np.histogram(ref, bins=edges)
        ch, _ = np.histogram(cur, bins=edges)
        rp = np.clip(rh/max(rh.sum(), 1), 1e-6, None)
        cp = np.clip(ch/max(ch.sum(), 1), 1e-6, None)
        return float(np.sum((cp-rp)*np.log(cp/rp)))

    feature_drift = pd.DataFrame([
        {'feature': col, 'psi': psi(train_rows[col], test_rows[col])}
        for col in feature_cols
    ]).sort_values('psi', ascending=False)
    feature_drift.to_csv(OUTPUT_DIRS['monitoring'] / 'feature_drift.csv', index=False)

    fig, ax = plt.subplots(figsize=(12, 6))
    top20_drift = feature_drift.head(20)
    colors_drift = ['red' if p > 0.2 else 'orange' if p > 0.1 else 'green' for p in top20_drift['psi']]
    ax.barh(top20_drift['feature'][::-1], top20_drift['psi'][::-1], color=colors_drift[::-1], edgecolor='white')
    ax.axvline(0.1, color='orange', ls='--', alpha=0.7, label='PSI=0.1 (moderate)')
    ax.axvline(0.2, color='red',    ls='--', alpha=0.7, label='PSI=0.2 (high drift)')
    ax.set_xlabel('PSI Score')
    ax.set_title('Feature Drift (PSI) -- Top 20', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9)
    save_plot('research_10_feature_drift.png')

    # [11] Prediction Distribution by Class
    print('[11/28] Prediction Distribution...')
    fig, ax = plt.subplots(figsize=(10, 6))
    tp = best_test_probs[:len(y_test)]
    ax.hist(tp[y_test.values==1], bins=60, alpha=0.65, density=True, color=PAPER_COLORS[2], label='Actual UP',   edgecolor='white')
    ax.hist(tp[y_test.values==0], bins=60, alpha=0.65, density=True, color=PAPER_COLORS[3], label='Actual DOWN', edgecolor='white')
    ax.axvline(best_threshold, color='red', ls='--', lw=2, label=f'Threshold={best_threshold:.3f}')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Density')
    ax.set_title('Prediction Distribution by Actual Class', fontsize=14, fontweight='bold')
    ax.legend()
    save_plot('research_11_prediction_distribution.png')

    # [12] Actual vs Predicted Scatter
    print('[12/28] Actual vs Predicted Scatter...')
    actual_close = test_rows['future_close'].values
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(actual_close, predicted_next_close, alpha=0.12, s=6, color=PAPER_COLORS[0], edgecolors='none')
    lims = [min(actual_close.min(), predicted_next_close.min()), max(actual_close.max(), predicted_next_close.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect')
    r2 = r2_score(actual_close, predicted_next_close)
    ax.text(0.05, 0.92, f'$R^2 = {r2:.4f}$', transform=ax.transAxes, fontsize=14, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_xlabel('Actual Next-Day Close ($)')
    ax.set_ylabel('Predicted ($)')
    ax.set_title(f'Regression: Actual vs Predicted -- {best_reg_name}', fontsize=14, fontweight='bold')
    ax.legend()
    save_plot('research_12_actual_vs_predicted.png')

    # [13] Residual Analysis
    print('[13/28] Residual Analysis...')
    residuals = actual_close - predicted_next_close
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(residuals, bins=80, color=PAPER_COLORS[0], alpha=0.7, edgecolor='white', density=True)
    axes[0].axvline(0, color='red', ls='--', lw=1.5)
    axes[0].set_xlabel('Residual ($)')
    axes[0].set_title('(a) Residual Distribution', fontweight='bold')
    axes[1].scatter(predicted_next_close, residuals, alpha=0.08, s=4, color=PAPER_COLORS[1], edgecolors='none')
    axes[1].axhline(0, color='red', ls='--', lw=1.5)
    axes[1].set_xlabel('Predicted ($)')
    axes[1].set_ylabel('Residual ($)')
    axes[1].set_title('(b) Residuals vs Fitted', fontweight='bold')
    sorted_res  = np.sort(residuals[~np.isnan(residuals)])
    norm_quant  = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_res)))
    axes[2].scatter(norm_quant, sorted_res[:len(norm_quant)], alpha=0.3, s=5, color=PAPER_COLORS[2])
    axes[2].plot([norm_quant.min(), norm_quant.max()], [norm_quant.min(), norm_quant.max()], 'r--', lw=1.5)
    axes[2].set_xlabel('Theoretical Quantiles')
    axes[2].set_ylabel('Sample Quantiles')
    axes[2].set_title('(c) Q-Q Plot', fontweight='bold')
    save_plot('research_13_residual_analysis.png')

    # [14] Class Distribution
    print('[14/28] Class Distribution...')
    fig, ax = plt.subplots(figsize=(10, 6))
    split_data = pd.DataFrame({
        'Split': ['Train', 'Train', 'Validation', 'Validation', 'Test', 'Test'],
        'Class': ['DOWN (0)', 'UP (1)']*3,
        'Count': [int((y_train==0).sum()), int((y_train==1).sum()),
                  int((y_val==0).sum()),   int((y_val==1).sum()),
                  int((y_test==0).sum()),  int((y_test==1).sum())]
    })
    pivot = split_data.pivot(index='Split', columns='Class', values='Count')
    pivot = pivot.reindex(['Train', 'Validation', 'Test'])
    pivot.plot(kind='bar', ax=ax, color=[PAPER_COLORS[3], PAPER_COLORS[2]], edgecolor='white', width=0.7)
    for c in ax.containers:
        ax.bar_label(c, label_type='center', fontsize=10, fontweight='bold', color='white')
    ax.set_title('Class Distribution Across Splits', fontsize=14, fontweight='bold')
    ax.set_xlabel('Data Split')
    ax.legend(title='Class')
    ax.tick_params(axis='x', rotation=0)
    save_plot('research_14_class_distribution.png')

    # [15] Radar Chart
    print('[15/28] Radar Chart...')
    try:
        radar_data = {}
        for name, info in model_registry.items():
            tp   = info['test_probs'][:len(y_test)]
            pred = (tp >= best_threshold).astype(int)
            radar_data[name] = [
                roc_auc_score(y_test[:len(tp)], tp),
                f1_score(y_test, pred, zero_division=0),
                (matthews_corrcoef(y_test, pred)+1)/2,
                balanced_accuracy_score(y_test, pred),
                average_precision_score(y_test[:len(tp)], tp),
            ]
        labels = ['ROC-AUC', 'F1', 'MCC (norm)', 'Bal. Acc', 'PR-AUC']
        N      = len(labels)
        angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
        angles += angles[:1]
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
        for i, (name, vals) in enumerate(radar_data.items()):
            v  = vals + vals[:1]
            lw = 2.5 if name==best_name else 1.5
            ax.plot(angles, v, '-o', label=name, lw=lw, markersize=5, color=PAPER_COLORS[i%len(PAPER_COLORS)])
            ax.fill(angles, v, alpha=0.06, color=PAPER_COLORS[i%len(PAPER_COLORS)])
        ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11)
        ax.set_ylim(0, 1)
        ax.set_title('Multi-Metric Radar Chart', fontsize=14, fontweight='bold', y=1.08)
        ax.legend(bbox_to_anchor=(1.3, 1.1), fontsize=9)
        save_plot('research_15_radar_chart.png')
    except Exception as e:
        print(f'  Radar chart failed: {e}')

    # [16] Walk-Forward Performance
    print('[16/28] Walk-Forward CV Bar...')
    if wf_results:
        wf_plot_df = pd.DataFrame(wf_results)
        fig, axes  = plt.subplots(2, 3, figsize=(18, 10))
        metrics_wf = [('val_auc', 'ROC-AUC', PAPER_COLORS[0]),
                      ('val_f1', 'F1 Score', PAPER_COLORS[1]),
                      ('val_mcc', 'MCC', PAPER_COLORS[2]),
                      ('val_ic', 'IC (Spearman)', PAPER_COLORS[3]),
                      ('val_sharpe', 'Sharpe Ratio', PAPER_COLORS[4]),
                      ('val_cum_return', 'Cum. Return', PAPER_COLORS[5])]
        for ax, (col, lbl, clr) in zip(axes.flat, metrics_wf):
            vals = wf_plot_df[col].values
            bars = ax.bar(wf_plot_df['fold'], vals, color=clr, edgecolor='white', alpha=0.85)
            ax.axhline(vals.mean(), color='black', ls='--', lw=2, label=f'Mean={vals.mean():.4f}')
            ax.axhline(0, color='gray', ls=':', alpha=0.4)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(vals.max()-vals.min())*0.02,
                        f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
            ax.set_xlabel('Fold')
            ax.set_ylabel(lbl)
            ax.set_title(f'Walk-Forward: {lbl}', fontweight='bold')
            ax.legend(fontsize=9)
        save_plot('research_16_walk_forward_cv.png')

    # [17] Volatility Regime Performance
    print('[17/28] Regime-Conditional Performance...')
    try:
        if regime_perf and not long_df.empty:
            reg_names  = list(regime_perf.keys())
            reg_sharpe = [regime_perf[r].get('sharpe_ratio', 0) for r in reg_names]
            reg_cum    = [regime_perf[r].get('cumulative_return', 0) for r in reg_names]
            reg_dd     = [regime_perf[r].get('max_drawdown', 0) for r in reg_names]
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            c_map = [PAPER_COLORS[2], PAPER_COLORS[0], PAPER_COLORS[3]][:len(reg_names)]
            
            axes[0].bar(reg_names, reg_sharpe, color=c_map, edgecolor='white')
            axes[0].axhline(0, color='gray', ls=':', alpha=0.5)
            axes[0].set_title('(a) Sharpe Ratio by Regime', fontweight='bold')
            axes[0].set_ylabel('Sharpe Ratio')
            
            axes[1].bar(reg_names, [r*100 for r in reg_cum], color=c_map, edgecolor='white')
            axes[1].axhline(0, color='gray', ls=':', alpha=0.5)
            axes[1].set_title('(b) Cumulative Return by Regime', fontweight='bold')
            axes[1].set_ylabel('Cumulative Return (%)')
            
            axes[2].bar(reg_names, [r*100 for r in reg_dd], color=c_map, edgecolor='white')
            axes[2].set_title('(c) Max Drawdown by Regime', fontweight='bold')
            axes[2].set_ylabel('Max Drawdown (%)')
            
            for ax in axes:
                ax.tick_params(axis='x', rotation=15)
            fig.suptitle('Strategy Performance by Market Regime', fontsize=14, fontweight='bold', y=1.02)
            save_plot('research_17_regime_performance.png')
    except Exception as e:
        print(f'  Regime performance failed: {e}')

    # [18] Prediction Confidence Deciles
    print('[18/28] Prediction Confidence Deciles...')
    try:
        conf_df = pd.DataFrame({
            'prob':   best_test_probs[:len(y_test)],
            'actual': y_test.values[:len(best_test_probs)],
            'ret':    test_rows['future_return'].values[:len(best_test_probs)]
        })
        conf_df['decile'] = pd.qcut(conf_df['prob'], 10, labels=False, duplicates='drop')
        dec_stats = conf_df.groupby('decile').agg(
            avg_prob=('prob', 'mean'),
            avg_ret =('ret', 'mean'),
            count   =('actual', 'count'),
        ).reset_index()
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        ax1.bar(dec_stats['decile'], dec_stats['avg_prob'], color=PAPER_COLORS[0], edgecolor='white', alpha=0.8)
        ax1.set_xlabel('Confidence Decile')
        ax1.set_ylabel('Avg Predicted Prob')
        ax1.set_title('(a) Average Confidence by Decile', fontweight='bold')
        
        col_rets = [PAPER_COLORS[2] if r > 0 else PAPER_COLORS[3] for r in dec_stats['avg_ret']]
        ax2.bar(dec_stats['decile'], dec_stats['avg_ret']*10000, color=col_rets, edgecolor='white', alpha=0.8)
        ax2.axhline(0, color='gray', ls=':', alpha=0.5)
        ax2.set_xlabel('Confidence Decile')
        ax2.set_ylabel('Avg Forward Return (bps)')
        ax2.set_title('(b) Forward Return by Confidence Decile', fontweight='bold')
        
        fig.suptitle(f'Confidence Analysis -- {best_name}', fontsize=14, fontweight='bold', y=1.02)
        save_plot('research_18_confidence_deciles.png')
    except Exception as e:
        print(f'  Confidence deciles failed: {e}')

    # [19] Drawdown Duration Distribution
    print('[19/28] Drawdown Duration Distribution...')
    try:
        dd_series = long_df['drawdown'].values if not long_df.empty else np.array([])
        in_dd     = dd_series < 0
        durations, depths = [], []
        start = None
        for i in range(len(in_dd)):
            if in_dd[i] and start is None:
                start = i
            elif not in_dd[i] and start is not None:
                durations.append(i - start)
                depths.append(dd_series[start:i].min())
                start = None
        if start is not None:
            durations.append(len(in_dd)-start)
            depths.append(dd_series[start:].min())
            
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        if durations:
            ax1.hist(durations, bins=min(30, len(durations)), color=PAPER_COLORS[3], alpha=0.7, edgecolor='white')
            ax1.axvline(np.median(durations), color='red', ls='--', lw=2, label=f'Median={np.median(durations):.0f}d')
            ax1.set_xlabel('Drawdown Duration (Days)')
            ax1.set_ylabel('Frequency')
            ax1.set_title('(a) Drawdown Duration Distribution', fontweight='bold')
            ax1.legend()
            
            ax2.scatter(durations, [d*100 for d in depths], alpha=0.6, s=40, color=PAPER_COLORS[1], edgecolors='white')
            ax2.set_xlabel('Duration (Days)')
            ax2.set_ylabel('Max Depth (%)')
            ax2.set_title('(b) Drawdown Depth vs Duration', fontweight='bold')
        
        fig.suptitle('Drawdown Analysis', fontsize=14, fontweight='bold', y=1.02)
        save_plot('research_19_drawdown_distribution.png')
    except Exception as e:
        print(f'  Drawdown distribution failed: {e}')

    # [20] Return Distribution + Tail Risk
    print('[20/28] Return Distribution + Tail Risk...')
    try:
        strat_rets = long_df['strat_ret'].dropna().values if not long_df.empty else np.array([0.0])
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        ax1.hist(strat_rets*100, bins=80, density=True, alpha=0.7, color=PAPER_COLORS[0], edgecolor='white', label='Strategy')
        x_r   = np.linspace(strat_rets.min()*100, strat_rets.max()*100, 200)
        mu, sg = strat_rets.mean()*100, strat_rets.std()*100
        ax1.plot(x_r, stats.norm.pdf(x_r, mu, sg), 'r-', lw=2, label='Normal')
        try:
            df_t, loc_t, sc_t = stats.t.fit(strat_rets*100)
            ax1.plot(x_r, stats.t.pdf(x_r, df_t, loc_t, sc_t), 'g--', lw=2, label=f'Student-t (df={df_t:.1f})')
        except Exception:
            pass
        cvar_val  = cvar(strat_rets)
        ax1.axvline(cvar_val*100, color='purple', ls=':', lw=2, label=f'CVaR 5%={cvar_val:.2%}')
        textstr   = f'Skew: {stats.skew(strat_rets):.3f}\nKurt: {stats.kurtosis(strat_rets):.3f}\nCVaR 5%: {cvar_val:.2%}'
        ax1.text(0.02, 0.95, textstr, transform=ax1.transAxes, fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax1.set_xlabel('Daily Return (%)')
        ax1.set_ylabel('Density')
        ax1.set_title('(a) Return Distribution + Tail Risk', fontweight='bold')
        ax1.legend(fontsize=9)
        
        sorted_s = np.sort(strat_rets)
        theo     = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_s)))
        ax2.scatter(theo, sorted_s[:len(theo)]*100, alpha=0.4, s=5, color=PAPER_COLORS[2])
        ax2.plot([theo.min(), theo.max()], [theo.min()*sg+mu, theo.max()*sg+mu], 'r--', lw=1.5)
        ax2.set_xlabel('Theoretical Normal Quantiles')
        ax2.set_ylabel('Sample Return (%)')
        ax2.set_title('(b) QQ Plot vs Normal', fontweight='bold')
        save_plot('research_20_return_distribution.png')
    except Exception as e:
        print(f'  Return distribution failed: {e}')

    # [21] IC Rolling Analysis + Information Ratio
    print('[21/28] Rolling IC + Information Ratio...')
    try:
        test_probs_arr  = best_test_probs[:len(test_rows)]
        test_rets_arr   = test_rows['future_return'].fillna(0).values[:len(test_probs_arr)]
        test_dates_arr  = test_rows['Date'].values[:len(test_probs_arr)]
        roll_ics        = []
        roll_window     = 60
        for i in range(roll_window, len(test_probs_arr)):
            ic_w = information_coefficient(test_probs_arr[i-roll_window:i], test_rets_arr[i-roll_window:i])
            roll_ics.append((test_dates_arr[i], ic_w))
            
        if roll_ics:
            roll_ic_df = pd.DataFrame(roll_ics, columns=['Date', 'IC'])
            roll_ic_df['Date'] = pd.to_datetime(roll_ic_df['Date'])
            roll_ic_df['IR']   = roll_ic_df['IC'].rolling(20).apply(lambda x: information_ratio(x), raw=True)
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
            
            ax1.plot(roll_ic_df['Date'], roll_ic_df['IC'], color=PAPER_COLORS[0], lw=1.5)
            ax1.fill_between(roll_ic_df['Date'], roll_ic_df['IC'], 0, where=roll_ic_df['IC']>0, alpha=0.2, color='green')
            ax1.fill_between(roll_ic_df['Date'], roll_ic_df['IC'], 0, where=roll_ic_df['IC']<0, alpha=0.2, color='red')
            ax1.axhline(0, color='gray', ls=':', alpha=0.5)
            ax1.axhline(roll_ic_df['IC'].mean(), color='blue', ls='--', lw=1.5, label=f'Mean IC={roll_ic_df["IC"].mean():.4f}')
            ax1.set_ylabel('Spearman IC')
            ax1.set_title(f'Rolling {roll_window}-Day IC -- {best_name}', fontweight='bold')
            ax1.legend(fontsize=9)
            
            ax2.plot(roll_ic_df['Date'], roll_ic_df['IR'], color=PAPER_COLORS[1], lw=1.5)
            ax2.axhline(0,   color='gray',  ls=':', alpha=0.5)
            ax2.axhline(0.5, color='green', ls='--', alpha=0.5, label='IR=0.5 (good)')
            ax2.set_ylabel('Information Ratio (IC/IC_std)')
            ax2.set_title('Rolling 20-Day Information Ratio', fontweight='bold')
            ax2.legend(fontsize=9)
            ax2.tick_params(axis='x', rotation=30)
            save_plot('research_21_rolling_ic_ir.png')
    except Exception as e:
        print(f'  Rolling IC failed: {e}')

    # [22] Signal Decay (IC vs Horizon)
    print('[22/28] Signal Decay (All Models)...')
    if decay_results:
        fig, ax = plt.subplots(figsize=(12, 6))
        for i, (name, ics) in enumerate(decay_results.items()):
            lw = 2.5 if name==best_name else 1.5
            ax.plot(horizons, ics, 'o-', color=PAPER_COLORS[i%len(PAPER_COLORS)], lw=lw, label=name, markersize=7)
        ax.axhline(0, color='gray', ls=':', alpha=0.5)
        ax.set_xlabel('Forecast Horizon (Days)')
        ax.set_ylabel('IC (Spearman)')
        ax.set_title('Signal Decay -- IC vs Forecast Horizon (All Models)', fontsize=14, fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xticks(horizons)
        save_plot('research_22_signal_decay_all_models.png')

    # [23] Cumulative IC (CIC)
    print('[23/28] Cumulative IC...')
    try:
        cum_ic  = np.cumsum([ic_val for _, ic_val in roll_ics]) if roll_ics else []
        cum_dates = [d for d, _ in roll_ics]
        if len(cum_ic) > 0 : 
            fig, ax = plt.subplots(figsize=(14, 5))
            ax.plot(cum_dates, cum_ic, color=PAPER_COLORS[4], lw=2)
            ax.fill_between(cum_dates, 0, cum_ic, where=np.array(cum_ic)>0, alpha=0.15, color='green')
            ax.fill_between(cum_dates, 0, cum_ic, where=np.array(cum_ic)<0, alpha=0.15, color='red')
            ax.axhline(0, color='gray', ls=':', alpha=0.5)
            ax.set_xlabel('Date')
            ax.set_ylabel('Cumulative IC')
            ax.set_title(f'Cumulative IC -- {best_name}', fontsize=14, fontweight='bold')
            ax.tick_params(axis='x', rotation=30)
            save_plot('research_23_cumulative_ic.png')
    except Exception as e:
        print(f'  Cumulative IC failed: {e}')

    # [24] Omega Ratio & CVaR Across Strategies
    print('[24/28] Risk Metrics Comparison (CVaR, Omega)...')
    try:
        strat_list = [
            ('Long-Only', long_df, long_metrics),
            ('Long-Short', ls_df, ls_metrics),
            ('Signal', signal_df, signal_metrics),
        ]
        categories  = ['Sharpe', 'Sortino', 'Calmar', 'Omega', 'CVaR (neg)']
        fig, ax = plt.subplots(figsize=(12, 6))
        x   = np.arange(len(categories)); w = 0.25
        for j, (sname, sdf, sm) in enumerate(strat_list):
            if sdf.empty:
                continue
            cv_val = cvar(sdf['strat_ret'].values)
            om_val = omega_ratio(sdf['strat_ret'].values)
            vals   = [sm.get('sharpe_ratio', 0),
                      sm.get('sortino_ratio', 0),
                      sm.get('calmar_ratio', 0),
                      min(om_val, 5.0),
                      -cv_val * 252]   # annualized CVaR (negative = bad)
            bars = ax.bar(x + j*w, vals, w, label=sname, color=PAPER_COLORS[j], edgecolor='white', alpha=0.85)
        ax.set_xticks(x + w)
        ax.set_xticklabels(categories)
        ax.set_title('Risk-Adjusted Metrics Comparison (Sharpe / Sortino / Calmar / Omega / CVaR)', fontsize=13, fontweight='bold')
        ax.legend(fontsize=9)
        ax.axhline(0, color='gray', ls=':', alpha=0.4)
        save_plot('research_24_risk_metrics_comparison.png')
    except Exception as e:
        print(f'  Risk metrics comparison failed: {e}')

    # [25] DL Training Dynamics (already saved earlier, reference copy)
    print('[25/28] DL Architecture Comparison...')
    if CONFIG['enable_dl_models']:
        dl_names = [n for n in ['AttentionLSTM', 'DilatedCausalCNN', 'TemporalTransformer'] if n in model_registry]
        if dl_names:
            fig, ax = plt.subplots(figsize=(12, 6))
            for i, name in enumerate(dl_names):
                hist = model_registry[name].get('history', {})
                if hist and 'val_auc' in hist:
                    ep = range(1, len(hist['val_auc'])+1)
                    lw = 2.5 if name==best_name else 1.5
                    ax.plot(ep, hist['val_auc'], '-', color=PAPER_COLORS[i], lw=lw, label=f'{name} (best={max(hist["val_auc"]):.4f})')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Val AUC')
            ax.set_title('DL Model Validation AUC During Training', fontsize=14, fontweight='bold')
            ax.legend(fontsize=10)
            save_plot('research_25_dl_val_auc.png')

    # [26] Factor Exposure: Alpha Decomposition
    print('[26/28] Factor Exposure (Alpha vs Beta)...')
    try:
        if not long_df.empty and 'mkt_ret' in long_df.columns:
            X_factor = np.column_stack([np.ones(len(long_df)), long_df['mkt_ret'].fillna(0).values])
            y_factor = long_df['strat_ret'].fillna(0).values
            coefs, _, _, _ = np.linalg.lstsq(X_factor, y_factor, rcond=None)
            alpha_daily = coefs[0]
            beta        = coefs[1]
            residuals_f = y_factor - X_factor @ coefs

            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            axes[0].scatter(long_df['mkt_ret'].values, long_df['strat_ret'].values, alpha=0.3, s=6, color=PAPER_COLORS[0])
            x_line = np.linspace(long_df['mkt_ret'].min(), long_df['mkt_ret'].max(), 100)
            axes[0].plot(x_line, alpha_daily + beta*x_line, 'r-', lw=2, label=f'Alpha={alpha_daily*252:.2%} ann. | Beta={beta:.2f}')
            axes[0].set_xlabel('Market Return')
            axes[0].set_ylabel('Strategy Return')
            axes[0].set_title('(a) Alpha/Beta Decomposition (Long-Only)', fontweight='bold')
            axes[0].legend(fontsize=10)

            axes[1].hist(residuals_f*100, bins=60, density=True, color=PAPER_COLORS[4], alpha=0.7, edgecolor='white', label='Idiosyncratic Returns')
            x_r2 = np.linspace(residuals_f.min()*100, residuals_f.max()*100, 200)
            axes[1].plot(x_r2, stats.norm.pdf(x_r2, 0, residuals_f.std()*100), 'r-', lw=2, label='Normal')
            axes[1].set_xlabel('Idiosyncratic Return (%)')
            axes[1].set_ylabel('Density')
            axes[1].set_title(f'(b) Idiosyncratic Return Distribution\nAnnualized Alpha: {alpha_daily*252:.2%}', fontweight='bold')
            axes[1].legend(fontsize=9)
            save_plot('research_26_alpha_decomposition.png')
    except Exception as e:
        print(f'  Alpha decomposition failed: {e}')

    # [27] Regime-Conditional Equity Curves
    print('[27/28] Regime-Conditional Equity Curves...')
    try:
        if ('regime_name' in test_rows.columns and not long_df.empty and 'Date' in long_df.columns):
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            regime_list  = sorted(test_rows['regime_name'].unique())
            regime_colors2 = {r: c for r, c in zip(regime_list, PAPER_COLORS[:3])}
            long_df2 = long_df.copy()
            long_df2['Date'] = pd.to_datetime(long_df2['Date'])
            test_regime_map = (test_rows[['Date', 'regime_name']].drop_duplicates('Date').set_index('Date')['regime_name'])
            long_df2['regime_name'] = long_df2['Date'].map(test_regime_map).fillna('Unknown')
            for ax, regime in zip(axes, regime_list):
                r_df = long_df2[long_df2['regime_name']==regime]
                if r_df.empty:
                    continue
                r_cum = (1 + r_df['strat_ret']).cumprod()
                ax.plot(r_df['Date'], r_cum, color=regime_colors2[regime], lw=2)
                ax.fill_between(r_df['Date'], 1, r_cum, where=r_cum>1, alpha=0.15, color='green')
                ax.fill_between(r_df['Date'], 1, r_cum, where=r_cum<1, alpha=0.15, color='red')
                ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
                sh_r = (r_df['strat_ret'].mean()/(r_df['strat_ret'].std()+1e-9))*np.sqrt(252)
                ax.set_title(f'{regime}\nSharpe={sh_r:.2f}', fontweight='bold')
                ax.set_ylabel('Cumulative Return')
                ax.tick_params(axis='x', rotation=25)
            save_plot('research_27_regime_equity_curves.png')
    except Exception as e:
        print(f'  Regime equity curves failed: {e}')

    # [28] Summary Performance Dashboard
    print('[28/28] Final Summary Dashboard...')
    try:
        fig = plt.figure(figsize=(20, 12))
        gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

        # Top row: equity curves
        ax0 = fig.add_subplot(gs[0, :2])
        if not long_df.empty:
            ax0.plot(long_df['Date'], long_df['cum_strat'], color='teal', lw=2, label='Long-Only')
            ax0.plot(long_df['Date'], long_df['cum_mkt'],   color='gray', lw=1.5, alpha=0.7, label='SPY')
            ax0.axhline(1.0, color='black', ls=':', alpha=0.3)
            ax0.set_title('Equity Curves', fontweight='bold')
            ax0.legend(fontsize=9)
            ax0.set_ylabel('Cum. Return')

        ax1 = fig.add_subplot(gs[0, 2:])
        cm2  = confusion_matrix(y_test[:len(test_preds)], test_preds)
        sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=False)
        ax1.set_title(f'Confusion Matrix -- {best_name}', fontweight='bold')
        ax1.set_xticklabels(['DOWN', 'UP'])
        ax1.set_yticklabels(['DOWN', 'UP'])

        # Middle row: ROC + signal decay
        ax2 = fig.add_subplot(gs[1, :2])
        for i, (name, info) in enumerate(model_registry.items()):
            tp = info['test_probs'][:len(y_test)]
            fpr, tpr, _ = roc_curve(y_test[:len(tp)], tp)
            auc = roc_auc_score(y_test[:len(tp)], tp)
            lw  = 2.5 if name==best_name else 1.0
            ax2.plot(fpr, tpr, lw=lw, label=f'{name[:12]} ({auc:.3f})', color=PAPER_COLORS[i%len(PAPER_COLORS)])
        ax2.plot([0, 1], [0, 1], 'k--', alpha=0.4)
        ax2.legend(fontsize=7)
        ax2.set_title('ROC Curves', fontweight='bold')

        ax3 = fig.add_subplot(gs[1, 2:])
        if decay_results:
            for i, (name, ics) in enumerate(decay_results.items()):
                ax3.plot(horizons, ics, 'o-', color=PAPER_COLORS[i%len(PAPER_COLORS)], lw=1.8, label=name[:12], markersize=5)
            ax3.axhline(0, color='gray', ls=':', alpha=0.5)
            ax3.set_title('Signal Decay (IC)', fontweight='bold')
            ax3.legend(fontsize=7)
            ax3.set_xticks(horizons)

        # Bottom row: metrics table
        ax4 = fig.add_subplot(gs[2, :])
        ax4.axis('off')
        col_keys   = ['sharpe_ratio', 'sortino_ratio', 'calmar_ratio', 'cagr', 'max_drawdown', 'win_rate', 'omega_ratio', 'cvar_5pct']
        tbl_data   = [col_keys]
        for m in [long_metrics, ls_metrics, signal_metrics]:
            row = []
            for k in col_keys:
                v = m.get(k, 0)
                if k in {'max_drawdown', 'cvar_5pct', 'win_rate', 'cagr'}:
                    row.append(f'{v:.2%}')
                else:
                    row.append(f'{v:.3f}')
            tbl_data.append(row)
            
        tbl = ax4.table(cellText=tbl_data[1:], colLabels=tbl_data[0], rowLabels=['Long-Only', 'Long-Short', 'Signal'], cellLoc='center', loc='center')
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(10)
        tbl.scale(1.0, 1.8)
        ax4.set_title('Strategy Performance Summary', fontweight='bold', pad=12)

        fig.suptitle(f'ML Stock Prediction Pipeline -- Final Dashboard\n'
                     f'Best Model: {best_name}  |  Threshold: {best_threshold:.4f}  |  '
                     f'Features: {len(feature_cols)}',
                     fontsize=14, fontweight='bold', y=1.01)
        save_plot('research_28_final_dashboard.png')
    except Exception as e:
        print(f'  Final dashboard failed: {e}')

    n_plots = len(list(OUTPUT_DIRS['plots'].glob('*.png')))
    print(f'\nAll {n_plots} figures saved to: {OUTPUT_DIRS["plots"]}')


# =======================================================================
# Cell 15 - MODEL SAVING & PRODUCTION INFERENCE FUNCTION
# =======================================================================

with cell_timer('Model Saving & Inference'):

    MODEL_VERSION = (f'v2.0_{DATA_HASH}_'
                     f'{pd.Timestamp.now().strftime("%Y%m%d_%H%M")}')
    print(f'Model version: {MODEL_VERSION}')

    def save_model_artifact(name, info):
        safe_name = name.lower().replace(' ', '_')
        if info['type'] == 'dl':
            dl_specs = {
                'AttentionLSTM':    {'model_class':'AttentionLSTM', 'kwargs':{'input_size':len(feature_cols), 'hidden':128, 'layers':2, 'dropout':0.3, 'n_heads':4}},
                'DilatedCausalCNN': {'model_class':'DilatedCausalCNN', 'kwargs':{'input_size':len(feature_cols), 'seq_len':CONFIG['seq_len'], 'ch':64, 'dropout':0.3}},
                'TemporalTransformer': {'model_class':'TemporalTransformer', 'kwargs':{'input_size':len(feature_cols), 'seq_len':CONFIG['seq_len'], 'd_model':64, 'n_heads':4, 'n_layers':2, 'dropout':0.2}},
            }
            spec = dl_specs.get(name, {'model_class':name, 'kwargs':{}})
            path = OUTPUT_DIRS['models'] / f'{safe_name}.pt'
            torch.save({
                'model_state_dict': info['model'].state_dict(),
                'model_name':   name, 'model_type': 'dl',
                'model_class':  spec['model_class'],
                'model_kwargs': spec['kwargs'],
                'feature_cols': feature_cols,
                'threshold':    best_threshold,
                'seq_len':      CONFIG['seq_len'],
                'scaler':       scaler,
                'winsorize_bounds': winsorize_bounds,
                'calibrator':   info.get('calibrator'),
                'config':       CONFIG, 'version': MODEL_VERSION,
                'data_hash':    DATA_HASH,
            }, path)
            return path

        bundle = {
            'model':             info['model'],
            'model_name':        name,
            'model_type':        info['type'],
            'score_kind':        info.get('score_kind'),
            'feature_cols':      feature_cols,
            'feature_space':     info.get('feature_space', 'scaled'),
            'threshold':         best_threshold,
            'scaler':            scaler,
            'winsorize_bounds':  winsorize_bounds,
            'calibrator':        info.get('calibrator'),
            'config':            CONFIG,
            'version':           MODEL_VERSION,
            'data_hash':         DATA_HASH,
        }
        path = OUTPUT_DIRS['models'] / f'{safe_name}.pkl'
        save_pickle_artifact(path, bundle)
        if joblib is not None:
            joblib.dump(bundle, OUTPUT_DIRS['models'] / f'{safe_name}.joblib')
        return path

    saved_paths = {}
    for name, info in model_registry.items():
        saved_paths[name] = str(save_model_artifact(name, info))

    # Save best classifier separately
    if best_info['type'] == 'dl':
        best_path = OUTPUT_DIRS['models'] / 'best_classifier.pt'
        torch.save({
            'model_state_dict': best_model.state_dict(),
            'model_name': best_name, 'threshold': best_threshold,
            'model_class': best_name,
            'feature_cols': feature_cols, 'seq_len': CONFIG['seq_len'],
            'scaler': scaler, 'winsorize_bounds': winsorize_bounds,
            'calibrator': best_info.get('calibrator'),
            'config': CONFIG, 'version': MODEL_VERSION,
        }, best_path)
    else:
        best_path = OUTPUT_DIRS['models'] / 'best_classifier.pkl'
        save_pickle_artifact(best_path, {
            'model': best_model, 'model_name': best_name,
            'score_kind': best_info.get('score_kind'),
            'feature_cols': feature_cols, 'scaler': scaler,
            'winsorize_bounds': winsorize_bounds,
            'threshold': best_threshold,
            'calibrator': best_info.get('calibrator'),
            'config': CONFIG, 'version': MODEL_VERSION,
        })

    # Save best regressor
    best_reg_path = OUTPUT_DIRS['models'] / 'best_regressor.pkl'
    save_pickle_artifact(best_reg_path, {
        'model': best_reg_info['model'], 'model_name': best_reg_name,
        'feature_cols': feature_cols, 'scaler': scaler,
        'winsorize_bounds': winsorize_bounds,
        'config': CONFIG, 'version': MODEL_VERSION,
    })

    # ---- Production Inference Function --------------------------------
    def predict_live(new_data_df, classifier_bundle_path, regressor_bundle_path=None):
        import pickle
        import torch

        # Load bundles
        if str(classifier_bundle_path).endswith('.pt'):
            clf_bundle = torch.load(classifier_bundle_path, map_location='cpu', weights_only=False)
            is_dl = True
        else:
            with open(classifier_bundle_path, 'rb') as f:
                clf_bundle = pickle.load(f)
            is_dl = False

        feature_cols_  = clf_bundle['feature_cols']
        scaler_        = clf_bundle.get('scaler')
        bounds_        = clf_bundle.get('winsorize_bounds', {})
        threshold_     = clf_bundle['threshold']
        score_kind_    = clf_bundle.get('score_kind', 'predict_proba')
        calibrator_    = clf_bundle.get('calibrator')

        if is_dl:
            model_class_name = clf_bundle.get('model_class', clf_bundle.get('model_name'))
            model_kwargs = clf_bundle.get('model_kwargs', {})
            if model_class_name == 'AttentionLSTM':
                model_ = AttentionLSTM(**model_kwargs) if model_kwargs else AttentionLSTM(len(feature_cols_))
            elif model_class_name == 'DilatedCausalCNN':
                model_ = DilatedCausalCNN(**model_kwargs) if model_kwargs else DilatedCausalCNN(len(feature_cols_), clf_bundle.get('seq_len', 20))
            elif model_class_name == 'TemporalTransformer':
                model_ = TemporalTransformer(**model_kwargs) if model_kwargs else TemporalTransformer(len(feature_cols_), clf_bundle.get('seq_len', 20))
            else:
                raise ValueError(f"Unknown model class {model_class_name}")
            model_.load_state_dict(clf_bundle['model_state_dict'])
            model_.eval()
            seq_len = clf_bundle.get('seq_len', 20)
        else:
            model_ = clf_bundle['model']

        # Feature engineering on new data
        all_signals = []
        for ticker, grp in new_data_df.groupby('ticker', sort=False):
            if len(grp) < 60:
                continue
            try:
                feat_df = create_features(grp.reset_index(drop=True))
                feat_df = feat_df.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols_)
                if feat_df.empty:
                    continue

                if is_dl:
                    if len(feat_df) < seq_len:
                        continue
                    X_live = feat_df.iloc[-seq_len:][feature_cols_].copy()
                else:
                    X_live = feat_df.iloc[-1:][feature_cols_].copy()

                for col in feature_cols_:
                    lo, hi = bounds_.get(col, (-np.inf, np.inf))
                    X_live[col] = X_live[col].clip(lo, hi)

                if scaler_ is not None:
                    X_live_sc = scaler_.transform(X_live)
                else:
                    X_live_sc = X_live.values

                if is_dl:
                    X_tensor = torch.tensor(X_live_sc, dtype=torch.float32).unsqueeze(0)
                    with torch.no_grad():
                        logits = model_(X_tensor)
                    prob = float(torch.sigmoid(logits).cpu().numpy()[0])
                else:
                    X_live_df = pd.DataFrame(X_live_sc, columns=feature_cols_)
                    if score_kind_ == 'predict_proba':
                        prob = float(model_.predict_proba(X_live_df)[0, 1])
                    else:
                        raw  = float(model_.decision_function(X_live_df)[0])
                        prob = float(expit(raw))

                if calibrator_ is not None:
                    prob = float(np.clip(calibrator_.predict([prob]), 1e-6, 1-1e-6)[0])

                signal = int(prob >= threshold_)

                # Fractional Kelly sizing
                k_size = 0.0
                if signal:
                    win_rate = float(threshold_)
                    avg_w    = abs(feat_df['return'].clip(lower=0).mean())
                    avg_l    = abs(feat_df['return'].clip(upper=0).mean())
                    k_full   = compute_kelly(win_rate, avg_w + 1e-9, avg_l + 1e-9)
                    k_size   = k_full * clf_bundle['config'].get('kelly_fraction', 0.25)
                    k_size   = float(np.clip(k_size, 0, clf_bundle['config'].get('max_position_concentration', 0.1)))

                last_row = feat_df.iloc[-1:]
                all_signals.append({
                    'ticker': ticker,
                    'date':   str(last_row['Date'].values[0]),
                    'close':  float(last_row['Close'].values[0]),
                    'prob':   prob,
                    'signal': signal,
                    'kelly_size': k_size,
                })
            except Exception as e:
                print(f'  Inference failed for {ticker}: {e}')

        signals = pd.DataFrame(all_signals)
        if not signals.empty:
            signals = signals.sort_values('prob', ascending=False).reset_index(drop=True)
            signals['kelly_size'] = signals['kelly_size'].clip(upper=clf_bundle['config'].get('max_position_concentration', 0.1))
        return signals

    print(f'\nModels saved: {len(saved_paths)}')
    for name, path in saved_paths.items():
        print(f'  {name:<22s} -> {path}')
    print(f'\nPredict function ready: predict_live(new_data_df, "{best_path}")')

# =======================================================================
# Cell 16 - MONITORING (Drift Detection + Performance Tracking)
# =======================================================================

with cell_timer('Monitoring & Drift Detection'):

    pred_drift_psi = psi(best_val_probs[:len(y_val)], best_test_probs[:len(y_test)])

    val_preds_mon = (best_val_probs[:len(y_val)] >= best_threshold).astype(int)
    test_preds_mon = test_preds

    wf_ic_vals = [r['val_ic'] for r in wf_results] if wf_results else []
    ir_wf      = information_ratio(wf_ic_vals)

    monitoring = {
        'best_model':         best_name,
        'threshold':          best_threshold,
        'model_version':      MODEL_VERSION,
        'data_hash':          DATA_HASH,
        'prediction_drift_psi':      float(pred_drift_psi),
        'mean_feature_psi':          float(feature_drift['psi'].mean()),
        'features_drifted_above_0_2': int((feature_drift['psi'] > 0.2).sum()),
        'walk_forward_ir':           float(ir_wf),
        'walk_forward_folds':        len(wf_results),
        'walk_forward_mean_auc':     float(np.mean([r['val_auc'] for r in wf_results])) if wf_results else 0,
        'walk_forward_mean_ic':      float(np.mean(wf_ic_vals)) if wf_ic_vals else 0,
        'val_roc_auc':     float(roc_auc_score(y_val[:len(best_val_probs)], best_val_probs)),
        'test_roc_auc':    float(cls_metrics['roc_auc']),
        'test_mcc':        float(cls_metrics['mcc']),
        'test_f1':         float(cls_metrics['f1']),
        'test_ic':         float(information_coefficient(best_test_probs[:len(y_test)], test_rows['future_return'].fillna(0).values[:len(best_test_probs)])),
        'long_only':       long_metrics,
        'long_short':      ls_metrics,
        'signal_strategy': signal_metrics,
        'regime_performance': regime_perf,
    }

    save_json_artifact(OUTPUT_DIRS['monitoring'] / 'monitoring_summary.json', monitoring)

    # Alert logic
    alerts = []
    if pred_drift_psi > 0.2:
        alerts.append(f'HIGH prediction drift PSI={pred_drift_psi:.3f} > 0.2')
    if monitoring['features_drifted_above_0_2'] > 5:
        alerts.append(f'{monitoring["features_drifted_above_0_2"]} features with PSI > 0.2')
    if abs(monitoring['val_roc_auc'] - monitoring['test_roc_auc']) > 0.05:
        alerts.append(f'AUC drop: val={monitoring["val_roc_auc"]:.4f} test={monitoring["test_roc_auc"]:.4f}')
    if ir_wf < 0.3 and wf_results:
        alerts.append(f'Low Information Ratio: IR={ir_wf:.3f} (signal may be unstable)')

    if alerts:
        print('\nMONITORING ALERTS:')
        for a in alerts:
            print(f'  ALERT: {a}')
    else:
        print('\nNo monitoring alerts. Model appears production-ready.')

    save_json_artifact(OUTPUT_DIRS['monitoring'] / 'alerts.json', {'alerts': alerts})
    print(f'Monitoring summary saved. Drift PSI={pred_drift_psi:.3f}  IR={ir_wf:.3f}')

# =======================================================================
# Cell 17 - PIPELINE SUMMARY (Final Report)
# =======================================================================

with cell_timer('Pipeline Summary'):

    total_elapsed = time.time() - _pipeline_start

    print('\n' + '='*72)
    print('   STOCK ML PREDICTION PIPELINE v2.0 -- FINAL SUMMARY')
    print('='*72)
    print(f'  Model Version:       {MODEL_VERSION}')
    print(f'  Data Fingerprint:    {DATA_HASH}')
    print(f'  Device:              {CONFIG["device"]}')
    print(f'  Total Elapsed:       {total_elapsed/60:.1f} min')
    print(f'  Tickers:             {data["ticker"].nunique()}')
    print(f'  Features Selected:   {len(feature_cols)} / {len(all_feature_cols)}')
    print(f'  Date Range:          {data["Date"].min().date()} to {data["Date"].max().date()}')

    print(f'\n  {"DATA SPLITS":^48s}')
    print(f'  {"--"*24}')
    print(f'  {"Split":<15s} {"Rows":>10s} {"UP%":>7s} {"Date Range":>25s}')
    for split_name, split_df, split_y in [
        ('Train',      train_rows, y_train),
        ('Validation', val_rows,   y_val),
        ('Test',       test_rows,  y_test)]:
        d0 = split_df['Date'].min().date()
        d1 = split_df['Date'].max().date()
        print(f'  {split_name:<15s} {len(split_df):>10,d} {split_y.mean():>6.1%} '
              f'{str(d0)+" to "+str(d1):>25s}')

    print(f'\n  {"CLASSIFIER LEADERBOARD":^48s}')
    print(f'  {"--"*24}')
    print(f'  {"Model":<22s} {"Val AUC":>8s} {"Test AUC":>9s} {"Val IC":>8s} {"Score":>8s}')
    for n, info in sorted(model_registry.items(), key=lambda x: -x[1].get('selection_score',0)):
        # Dynamic check for test AUC
        current_test_probs = info['test_probs'][:len(y_test)]
        te_auc = roc_auc_score(y_test[:len(current_test_probs)], current_test_probs)
        marker = '  *' if n == best_name else ''
        print(f'  {n:<22s} {info["val_auc"]:>8.4f} {te_auc:>9.4f} '
              f'{info.get("val_ic",0):>8.4f} {info.get("selection_score",0):>8.4f}{marker}')

    print(f'\n  {"BEST CLASSIFIER":^48s}  ({best_name})')
    print(f'  {"--"*24}')
    print(f'  Threshold ({CONFIG["threshold_metric"]}):  {best_threshold:.4f}')
    for k in ['accuracy','balanced_accuracy','precision','recall','f1',
              'roc_auc','pr_auc','mcc','log_loss','brier_score']:
        if k in cls_metrics:
            print(f'  {k:<25s}: {cls_metrics[k]:.4f}')

    print(f'\n  Test IC:  {monitoring["test_ic"]:.4f}  |  Walk-forward IR: {ir_wf:.4f}')

    print(f'\n  {"STRATEGY COMPARISON":^48s}')
    print(f'  {"--"*24}')
    print(f'  {"Metric":<26s} {"Long-Only":>10s} {"Long-Short":>11s} {"Signal":>9s}')
    for k in ['sharpe_ratio','sortino_ratio','calmar_ratio','cagr',
              'max_drawdown','cvar_5pct','omega_ratio','win_rate','cumulative_return']:
        v1 = long_metrics.get(k,0)
        v2 = ls_metrics.get(k,0)
        v3 = signal_metrics.get(k,0)
        if k in {'cagr','max_drawdown','cvar_5pct','win_rate','cumulative_return'}:
            print(f'  {k:<26s} {v1:>9.2%} {v2:>10.2%} {v3:>8.2%}')
        else:
            print(f'  {k:<26s} {v1:>10.4f} {v2:>11.4f} {v3:>9.4f}')

    print(f'\n  {"WALK-FORWARD STABILITY":^48s}')
    print(f'  {"--"*24}')
    print(f'  Folds: {len(wf_results)}  |  Mean AUC: {monitoring["walk_forward_mean_auc"]:.4f}  |  Mean IC: {monitoring["walk_forward_mean_ic"]:.4f}')
    print(f'  Information Ratio (IC/IC_std): {ir_wf:.4f}')
    print(f'  Signal stability: {"GOOD (IR>0.5)" if ir_wf > 0.5 else "MARGINAL (0.3<IR<0.5)" if ir_wf > 0.3 else "UNSTABLE (IR<0.3)"}')

    print(f'\n  {"OUTPUT ARTIFACTS":^48s}')
    print(f'  {"--"*24}')
    for dir_name, dir_path in OUTPUT_DIRS.items():
        if dir_name == 'root': continue
        n_files = len(list(dir_path.iterdir())) if dir_path.exists() else 0
        print(f'  {dir_name:<15s}: {n_files:3d} files  ({dir_path})')

    n_research = len(list(OUTPUT_DIRS['plots'].glob('research_*.png')))
    print(f'\n  Research figures: {n_research} / 28')
    
    if alerts:
        print('\n  MONITORING ALERTS:')
        for a in alerts:
            print(f'    - {a}')
    else:
        print('\n  Status: PRODUCTION READY')

    print('\n' + '='*72)
    print('  Pipeline v2.0 complete.')

Config validation passed
Device:          cpu
Random seed:     42
Stocks path:     /kaggle/input/datasets/borismarjanovic/price-volume-data-for-all-us-stocks-etfs/Stocks
ETF path:        /kaggle/input/datasets/borismarjanovic/price-volume-data-for-all-us-stocks-etfs/ETFs
DL patience:     6
Walk-fwd folds:  5
Kelly fraction:  0.25
IC horizons:     [1, 2, 5, 10, 20]
SHAP available:  True
joblib:          True

  Data Loading & Cleaning started...
Loading stock files...
Loading ETF files...
Raw data shape:   (658779, 8)
Unique tickers:   210
Date range:       1970-01-05 to 2017-11-10
Data fingerprint: 848ebe4c42e97734
  Data Loading & Cleaning completed in 8.0s

  Feature Engineering started...
Processing features for 25 tickers...
After feature engineering: (178890, 118)  |  tickers: 25
  Feature Engineering completed in 4.7s

  Market Regime Detection started...
Regime distribution (unique trading days):
  High-Vol (Bear)       :  1077 days (8.9%)
  Low-Vol (Bull)        :  1161 days (9

PermutationExplainer explainer: 1501it [02:36,  9.21it/s]                          


  SHAP plots saved.
  Explainability & Feature Importance completed in 159.5s

  Core Visualizations started...
Core visualizations saved.
  Core Visualizations completed in 5.3s

  Research Paper Visualizations (28 Figures) started...
GENERATING 28 RESEARCH PAPER VISUALIZATIONS

[1/28] Feature Correlation Heatmap...
[2/28] Multi-Model Feature Importance...
[3/28] Learning Curves...
[4/28] Calibration Plot...
[5/28] Threshold Optimization...
[6/28] Precision-Recall-F1 Trade-off...
[7/28] Cumulative Returns (All Models)...
[8/28] Rolling Sharpe...
[9/28] Monthly Returns Heatmap...
[10/28] Feature Drift PSI...
[11/28] Prediction Distribution...
[12/28] Actual vs Predicted Scatter...
[13/28] Residual Analysis...
[14/28] Class Distribution...
[15/28] Radar Chart...
[16/28] Walk-Forward CV Bar...
[17/28] Regime-Conditional Performance...
[18/28] Prediction Confidence Deciles...
[19/28] Drawdown Duration Distribution...
[20/28] Return Distribution + Tail Risk...
[21/28] Rolling IC + Informat

In [7]:
# [23] Cumulative IC (CIC)
print('[23/28] Cumulative IC...')
try:
    cum_ic  = np.cumsum([ic_val for _, ic_val in roll_ics]) if roll_ics else []
    cum_dates = [d for d, _ in roll_ics]
    if len(cum_ic) > 0:
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.plot(cum_dates, cum_ic, color=PAPER_COLORS[4], lw=2)
        ax.fill_between(cum_dates, 0, cum_ic, where=np.array(cum_ic)>0, alpha=0.15, color='green')
        ax.fill_between(cum_dates, 0, cum_ic, where=np.array(cum_ic)<0, alpha=0.15, color='red')
        ax.axhline(0, color='gray', ls=':', alpha=0.5)
        ax.set_xlabel('Date')
        ax.set_ylabel('Cumulative IC')
        ax.set_title(f'Cumulative IC -- {best_name}', fontsize=14, fontweight='bold')
        ax.tick_params(axis='x', rotation=30)
        save_plot('research_23_cumulative_ic.png')
except Exception as e:
    print(f'  Cumulative IC failed: {e}')

[23/28] Cumulative IC...
